# B6 · Explicabilité, équité et éthique — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b6_explicabilite/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b6_explicabilite/03_notebook.ipynb`, créée par `python tools/start_chapter.py B6`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B6.Q1 à B6.Q12, B6.R1 à B6.R3, B6.1, B6.3 à B6.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B6.12 | Explications globales sur California : coefficients, permutation, dépendance partielle et ICE | 📦 | ★★ | 30 |
|  | B6.13 | Prédire la forme des courbes ICE du revenu médian et de la latitude | 🔮 | ★ | 10 |
|  | B6.14 | shapley_values : Shapley exact par énumération des coalitions | 🔨 | ★★★ | 45 |
|  | B6.15 | SHAP sur un gradient boosting California : TreeExplainer, waterfall, beeswarm, dépendance | 📦 | ★★ | 30 |
|  | B6.16 | Le fond change tout : trois fonds différents, puis une feature dupliquée | 🔬 | ★★★ | 40 |
|  | B6.17 | Penguins : SHAP par classe pour une régression logistique et une forêt aléatoire | 📦 | ★★ | 25 |
| B | B6.18 | integrated_gradients et grad_cam | 🔨 | ★★ | 30 |
|  | B6.19 | Saillance, gradient × entrée et gradients intégrés sur un CNN CIFAR-10 | 📦 | ★★ | 30 |
|  | B6.20 | Grad-CAM par hooks sur le ResNet fine-tuné du B1 : bonnes prédictions et erreurs | 📦 | ★★★ | 45 |
| B | B6.21 et B6.22 | un Grad-CAM à réparer, le test de santé d'Adebayo (*prochaine session de génération*) | | | |
| C | B6.23 à B6.29 | l'équité : tes indicateurs par groupe, l'audit d'Adult avec intervalles, l'effet du retrait du sexe, l'atténuation et son coût, la fiche modèle et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B6")  # your own library

## Objectifs et rappel express

- Calculer à la main des valeurs de Shapley exactes, des gradients intégrés et une carte Grad-CAM, et les relier à ta librairie (`explain.py`) et à shap.
- Expliquer un modèle tabulaire globalement (permutation, dépendance partielle, ICE) et localement (Shapley, waterfall), en disant quel fond a servi.
- Produire des cartes d'attribution sur un CNN et les soumettre à un test de santé.
- Auditer un classifieur groupe par groupe, avec des intervalles, essayer des corrections, chiffrer leur coût et documenter le tout dans une fiche modèle.

**Rappel express.** Shapley : $v(S) = \frac{1}{B}\sum_k f(\mathbf{x}_S, \mathbf{b}^{(k)}_{\bar S})$, $\phi_i = \sum_{S \not\ni i} \frac{|S|!\,(M - |S| - 1)!}{M!}\,[v(S \cup \{i\}) - v(S)]$, $\sum_i \phi_i = f(\mathbf{x}) - v(\varnothing)$ ; modèle linéaire : $\phi_i = w_i(x_i - \bar b_i)$. Gradients intégrés : $\operatorname{IG}_i = (x_i - x'_i)\int_0^1 \partial_i f(\mathbf{x}' + \alpha(\mathbf{x} - \mathbf{x}'))\,d\alpha$, de somme $f(\mathbf{x}) - f(\mathbf{x}')$. Grad-CAM : $\alpha_c$ = moyenne spatiale de $\partial y^k / \partial A^c$, carte $\operatorname{ReLU}(\sum_c \alpha_c A^c)$. Équité : $\Delta_{\text{DP}}$ = écart des taux de sélection, $\Delta_{\text{EO}}$ = le plus grand des écarts de TPR et de FPR ; $\operatorname{FPR} = \frac{p}{1-p}\cdot\frac{1 - \operatorname{PPV}}{\operatorname{PPV}}\cdot(1 - \operatorname{FNR})$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B6.1, B6.3, B6.4 et B6.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un vecteur : une liste de nombres (`[0.5, -1.25]`). Un taux s'écrit comme une proportion (`0.25`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B6.2 et B6.6, l'estimation 🧮 B6.7, la réflexion (🗣️ B6.8, ⚖️ B6.9 et B6.10, 📄 B6.11) et l'entretien se corrigent avec `05_solutions.md`.

**B6.Q1 — Interprétable par construction ou expliqué après coup ?**

In [ ]:
# B6.Q1: the values you found on paper
answer_B6_Q1a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q1b = ...  # b) True or False
answer_B6_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_Q1d = ...  # d) True or False
answer_B6_Q1e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B6_Q1a, answer_B6_Q1b, answer_B6_Q1c, answer_B6_Q1d, answer_B6_Q1e]):
    wb.check(f"B6.Q1{letter}", answer)

**B6.Q2 — Explication globale ou locale : quel outil répond à quelle question ?**

In [ ]:
# B6.Q2: the values you found on paper
answer_B6_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q2c = ...  # c) True or False
answer_B6_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_B6_Q2e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_B6_Q2a, answer_B6_Q2b, answer_B6_Q2c, answer_B6_Q2d, answer_B6_Q2e]):
    wb.check(f"B6.Q2{letter}", answer)

**B6.Q3 — Dépendance partielle et courbes ICE : ce que montre la moyenne, ce qu'elle cache**

In [ ]:
# B6.Q3: the values you found on paper
answer_B6_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q3b = ...  # b) a whole number
answer_B6_Q3c = ...  # c) a number, 2 decimals
answer_B6_Q3d = ...  # d) True or False
answer_B6_Q3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q3a, answer_B6_Q3b, answer_B6_Q3c, answer_B6_Q3d, answer_B6_Q3e]):
    wb.check(f"B6.Q3{letter}", answer)

**B6.Q4 — Les quatre propriétés de Shapley : efficacité, symétrie, joueur nul, additivité**

In [ ]:
# B6.Q4: the values you found on paper
answer_B6_Q4a = ...  # a) a number, 2 decimals
answer_B6_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q4c = ...  # c) a number, 1 decimal
answer_B6_Q4d = ...  # d) True or False
answer_B6_Q4e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B6_Q4a, answer_B6_Q4b, answer_B6_Q4c, answer_B6_Q4d, answer_B6_Q4e]):
    wb.check(f"B6.Q4{letter}", answer)

**B6.Q5 — Changer le fond (background) change les valeurs SHAP : vrai ou faux**

In [ ]:
# B6.Q5: the values you found on paper
answer_B6_Q5a = ...  # a) True or False
answer_B6_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q5c = ...  # c) a whole number
answer_B6_Q5d = ...  # d) a whole number
answer_B6_Q5e = ...  # e) True or False
answer_B6_Q5f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_B6_Q5a, answer_B6_Q5b, answer_B6_Q5c, answer_B6_Q5d, answer_B6_Q5e, answer_B6_Q5f]):
    wb.check(f"B6.Q5{letter}", answer)

**B6.Q6 — Features corrélées : ce que SHAP répartit et ce qu'il ne dit pas**

In [ ]:
# B6.Q6: the values you found on paper
answer_B6_Q6a = ...  # a) a whole number
answer_B6_Q6b = ...  # b) a whole number
answer_B6_Q6c = ...  # c) True or False
answer_B6_Q6d = ...  # d) the letter of your choice, e.g. "E"
answer_B6_Q6e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q6a, answer_B6_Q6b, answer_B6_Q6c, answer_B6_Q6d, answer_B6_Q6e]):
    wb.check(f"B6.Q6{letter}", answer)

**B6.Q7 — Gradient, gradient × entrée, gradients intégrés : pourquoi une ligne de référence ?**

In [ ]:
# B6.Q7: the values you found on paper
answer_B6_Q7a = ...  # a) a whole number
answer_B6_Q7b = ...  # b) a whole number
answer_B6_Q7c = ...  # c) a whole number
answer_B6_Q7d = ...  # d) True or False
answer_B6_Q7e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q7a, answer_B6_Q7b, answer_B6_Q7c, answer_B6_Q7d, answer_B6_Q7e]):
    wb.check(f"B6.Q7{letter}", answer)

**B6.Q8 — Grad-CAM : pourquoi la dernière couche convolutive ?**

In [ ]:
# B6.Q8: the values you found on paper
answer_B6_Q8a = ...  # a) a whole number
answer_B6_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q8c = ...  # c) a whole number
answer_B6_Q8d = ...  # d) True or False
answer_B6_Q8e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q8a, answer_B6_Q8b, answer_B6_Q8c, answer_B6_Q8d, answer_B6_Q8e]):
    wb.check(f"B6.Q8{letter}", answer)

**B6.Q9 — Une carte de saillance qui ne dépend pas du modèle : que prouve le test de randomisation ?**

In [ ]:
# B6.Q9: the values you found on paper
answer_B6_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q9b = ...  # b) True or False
answer_B6_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_Q9d = ...  # d) True or False
answer_B6_Q9e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B6_Q9a, answer_B6_Q9b, answer_B6_Q9c, answer_B6_Q9d, answer_B6_Q9e]):
    wb.check(f"B6.Q9{letter}", answer)

**B6.Q10 — Sources de biais : données, labels, variables de substitution, boucles de rétroaction**

In [ ]:
# B6.Q10: the values you found on paper
answer_B6_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q10c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_Q10d = ...  # d) True or False
answer_B6_Q10e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q10a, answer_B6_Q10b, answer_B6_Q10c, answer_B6_Q10d, answer_B6_Q10e]):
    wb.check(f"B6.Q10{letter}", answer)

**B6.Q11 — Parité démographique, égalité des chances, calibration : peut-on tout avoir ?**

In [ ]:
# B6.Q11: the values you found on paper
answer_B6_Q11a = ...  # a) a number, 2 decimals
answer_B6_Q11b = ...  # b) True or False
answer_B6_Q11c = ...  # c) True or False
answer_B6_Q11d = ...  # d) a number, 2 decimals
answer_B6_Q11e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q11a, answer_B6_Q11b, answer_B6_Q11c, answer_B6_Q11d, answer_B6_Q11e]):
    wb.check(f"B6.Q11{letter}", answer)

**B6.Q12 — AI Act : dans quelle catégorie de risque tombe un scoring de crédit ?**

In [ ]:
# B6.Q12: the values you found on paper
answer_B6_Q12a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_Q12d = ...  # d) True or False
answer_B6_Q12e = ...  # e) the letter of your choice, e.g. "E"
answer_B6_Q12f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B6_Q12a, answer_B6_Q12b, answer_B6_Q12c, answer_B6_Q12d, answer_B6_Q12e, answer_B6_Q12f]):
    wb.check(f"B6.Q12{letter}", answer)

**B6.R1 — Visualiser ce qu'a appris un CNN, et l'attaque FGSM**

In [ ]:
# B6.R1: the values you found on paper
answer_B6_R1a = ...  # a) a list of three numbers, 2 decimals
answer_B6_R1b = ...  # b) True or False
answer_B6_R1c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_R1d = ...  # d) a number, 2 decimals
answer_B6_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B6_R1a, answer_B6_R1b, answer_B6_R1c, answer_B6_R1d, answer_B6_R1e]):
    wb.check(f"B6.R1{letter}", answer)

**B6.R2 — feature_importances_ contre importance par permutation**

In [ ]:
# B6.R2: the values you found on paper
answer_B6_R2a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_R2b = ...  # b) True or False
answer_B6_R2c = ...  # c) a whole number
answer_B6_R2d = ...  # d) the letter of your choice, e.g. "E"
answer_B6_R2e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B6_R2a, answer_B6_R2b, answer_B6_R2c, answer_B6_R2d, answer_B6_R2e]):
    wb.check(f"B6.R2{letter}", answer)

**B6.R3 — TPR, FPR et precision à partir d'une matrice de confusion**

In [ ]:
# B6.R3: the values you found on paper
answer_B6_R3a = ...  # a) a number, 2 decimals
answer_B6_R3b = ...  # b) a number, 2 decimals
answer_B6_R3c = ...  # c) a number, 3 decimals
answer_B6_R3d = ...  # d) a number, 3 decimals
answer_B6_R3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_R3a, answer_B6_R3b, answer_B6_R3c, answer_B6_R3d, answer_B6_R3e]):
    wb.check(f"B6.R3{letter}", answer)

**Ex B6.1 — Valeurs de Shapley exactes à la main : trois features, huit coalitions**

In [ ]:
# Ex B6.1: the values you found on paper
answer_B6_1a = ...  # a) a whole number
answer_B6_1b = ...  # b) a whole number
answer_B6_1c = ...  # c) a whole number
answer_B6_1d = ...  # d) a number, 2 decimals
answer_B6_1e = ...  # e) a number, 1 decimal
answer_B6_1f = ...  # f) a whole number
answer_B6_1g = ...  # g) a number, 1 decimal
answer_B6_1h = ...  # h) True or False
answer_B6_1i = ...  # i) a whole number
answer_B6_1j = ...  # j) a number, 1 decimal
answer_B6_1k = ...  # k) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefghijk", [answer_B6_1a, answer_B6_1b, answer_B6_1c, answer_B6_1d, answer_B6_1e, answer_B6_1f, answer_B6_1g, answer_B6_1h, answer_B6_1i, answer_B6_1j, answer_B6_1k]):
    wb.check(f"B6.1{letter}", answer)

**Ex B6.3 — Gradients intégrés à la main sur f(x) = x₁² + 3x₂ et vérification de la complétude**

In [ ]:
# Ex B6.3: the values you found on paper
answer_B6_3a = ...  # a) a list of two numbers
answer_B6_3b = ...  # b) a list of two numbers
answer_B6_3c = ...  # c) a whole number
answer_B6_3d = ...  # d) a list of two numbers
answer_B6_3e = ...  # e) a number, 1 decimal
answer_B6_3f = ...  # f) a number, 1 decimal
answer_B6_3g = ...  # g) a number, 1 decimal
answer_B6_3h = ...  # h) a whole number
answer_B6_3i = ...  # i) a whole number
answer_B6_3j = ...  # j) a whole number
answer_B6_3k = ...  # k) a number, 2 decimals
answer_B6_3l = ...  # l) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefghijkl", [answer_B6_3a, answer_B6_3b, answer_B6_3c, answer_B6_3d, answer_B6_3e, answer_B6_3f, answer_B6_3g, answer_B6_3h, answer_B6_3i, answer_B6_3j, answer_B6_3k, answer_B6_3l]):
    wb.check(f"B6.3{letter}", answer)

**Ex B6.4 — Grad-CAM à la main : deux cartes 3×3 et leurs gradients**

In [ ]:
# Ex B6.4: the values you found on paper
answer_B6_4a = ...  # a) a number, 1 decimal
answer_B6_4b = ...  # b) a number, 1 decimal
answer_B6_4c = ...  # c) a number, 1 decimal
answer_B6_4d = ...  # d) a whole number
answer_B6_4e = ...  # e) a number, 2 decimals
answer_B6_4f = ...  # f) the letter of your choice, e.g. "E"
answer_B6_4g = ...  # g) a number, 2 decimals
answer_B6_4h = ...  # h) the letter of your choice, e.g. "E"
answer_B6_4i = ...  # i) True or False
answer_B6_4j = ...  # j) a whole number

for letter, answer in zip("abcdefghij", [answer_B6_4a, answer_B6_4b, answer_B6_4c, answer_B6_4d, answer_B6_4e, answer_B6_4f, answer_B6_4g, answer_B6_4h, answer_B6_4i, answer_B6_4j]):
    wb.check(f"B6.4{letter}", answer)

**Ex B6.5 — Audit à la main : deux groupes, deux matrices de confusion, cinq indicateurs**

In [ ]:
# Ex B6.5: the values you found on paper
answer_B6_5a = ...  # a) a number, 2 decimals
answer_B6_5b = ...  # b) a number, 3 decimals
answer_B6_5c = ...  # c) a number, 2 decimals
answer_B6_5d = ...  # d) a number, 3 decimals
answer_B6_5e = ...  # e) a number, 3 decimals
answer_B6_5f = ...  # f) a number, 2 decimals
answer_B6_5g = ...  # g) a number, 2 decimals
answer_B6_5h = ...  # h) a number, 2 decimals
answer_B6_5i = ...  # i) True or False
answer_B6_5j = ...  # j) True or False
answer_B6_5k = ...  # k) a whole number

for letter, answer in zip("abcdefghijk", [answer_B6_5a, answer_B6_5b, answer_B6_5c, answer_B6_5d, answer_B6_5e, answer_B6_5f, answer_B6_5g, answer_B6_5h, answer_B6_5i, answer_B6_5j, answer_B6_5k]):
    wb.check(f"B6.5{letter}", answer)

## Partie A · Modèles tabulaires : explications globales, valeurs de Shapley et SHAP

Tu expliques d'abord un gradient boosting de California **globalement** : coefficients d'une régression, importance par permutation, dépendance partielle et courbes ICE (B6.12), dont tu prédis la forme pour deux features (B6.13). Puis **localement** : tu écris le Shapley exact de ta librairie (B6.14), tu le compares à TreeSHAP sur un gradient boosting et tu lis les trois graphiques de shap (B6.15), tu mesures ce que changent le fond et une feature dupliquée (B6.16), et tu expliques un classifieur à trois classes de Penguins (B6.17). Au ch. 14, tu as entraîné des gradient boostings sur California et comparé `feature_importances_` à la permutation ; au ch. 15, tu as construit des pipelines : ici, tu ouvres la boîte.

La cellule suivante charge les outils de la partie : `wbx` (`wb.explain_data`, l'infrastructure du chapitre), le découpage figé de California (`X_TRAIN`, `Y_TRAIN` : 12 384 districts ; `X_VAL`, `Y_VAL` : 4 128 ; les mêmes à chaque exécution), `FEATURES` (les 8 noms de colonnes), `GBR`, le gradient boosting de référence (`HistGradientBoostingRegressor(random_state=0)` ajusté sur `X_TRAIN`), dont se servent les vérifications et les expériences données, `TEST_EXPLAIN` (le fichier des tests de `explain.py`) et `shap`. Les features de California y sont arrondies en float32 (B6.15 dit pourquoi).

In [ ]:
# Tools of part A: California and Penguins (frozen splits), the reference model of the checks, your library explain.py
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from wb import explain_data as wbx
from wb.nbtools import StopCheck, checked, filled, fr, pair, refuses, returned, run_pytest_group, same, verdict, written

warnings.filterwarnings("ignore", message=".*IProgress not found.*")              # tqdm, imported by shap
warnings.filterwarnings("ignore", message="X does not have valid feature names")   # a model fitted on a DataFrame,
                                                                                   # called on an array (shapley_values)
try:
    import shap
except ImportError:                       # shap is on Colab and in requirements.txt; wb.ensure("shap") installs it
    shap = None
    print("⏳ shap n'est pas installé : les cellules qui s'en servent le diront (wb.ensure('shap') l'installe).")

TEST_EXPLAIN = "tests/test_chb6_explain.py"            # the tests of your library explain.py
CAL = wbx.california()                                 # the frozen split of California (features rounded to float32)
FEATURES = CAL.features
X_TRAIN, Y_TRAIN, X_VAL, Y_VAL = CAL.X_train, CAL.y_train, CAL.X_val, CAL.y_val
GBR = HistGradientBoostingRegressor(random_state=0).fit(X_TRAIN, Y_TRAIN)   # the reference gradient boosting


def pct(x, digits=0):
    """A share written the French way: 0.734 -> '73 %'."""
    return f"{100 * x:.{digits}f} %".replace(".", ",")


print(f"California : {len(X_TRAIN)} districts d'entraînement, {len(X_VAL)} de validation, {len(CAL.X_test)} de test, "
      f"{len(FEATURES)} features ({', '.join(FEATURES)}) ; cible MedHouseVal (centaines de milliers de dollars). "
      f"Gradient boosting de référence : R² de validation {fr(GBR.score(X_VAL, Y_VAL))}.")

### Ex B6.12 — Explications globales sur California : coefficients, permutation, dépendance partielle et ICE 📦 ★★ ⏱️ 30 min
**Objectif :** comparer trois explications globales d'un même problème et savoir ce que chacune mesure.  
**Prérequis :** ch. 14 et 15 · fiche §206.2 (« Coefficients et arbres », « L'importance par permutation », « La dépendance partielle », « Les courbes ICE ») · 🔁 R2 · **Fil rouge :** California · **Parcours :** R, C

**Contrat** des cinq fonctions de la cellule suivante :
- `linear_12(X, y)` rend un `Pipeline` `StandardScaler` → `LinearRegression`, ajusté sur `(X, y)` ;
- `coefs_12(pipeline)` rend une `pd.Series` des 8 coefficients de la `LinearRegression` du pipeline, indexée par les features, dans l'ordre de `FEATURES` : l'effet d'**un écart-type** de chaque feature, les autres fixées ;
- `boosting_12(X, y)` rend un `HistGradientBoostingRegressor(random_state=0)`, tous les autres réglages par défaut, ajusté sur `(X, y)` ;
- `permutation_12(model, X, y)` rend une `pd.Series` : la baisse moyenne du R² de `permutation_importance(model, X, y, n_repeats=5, random_state=0)`, indexée par les colonnes de `X`, triée de la plus grande à la plus petite ;
- `pd_ice_12(model, X, feature)` rend le dictionnaire `{"grid": (50,), "average": (50,), "individual": (len(X), 50)}` de `partial_dependence(model, X, [feature], kind="both", grid_resolution=50)` (la grille va du 5ᵉ au 95ᵉ centile de la feature).

La cellule de vérification appelle `linear_12` et `boosting_12` sur `X_TRAIN`, `permutation_12` sur le gradient boosting de référence `GBR` et la **validation**, `pd_ice_12` sur `GBR`, 300 districts de validation et `AveOccup`, et compare chaque résultat à scikit-learn. Une fois les trois résultats justes, elle trace les coefficients en valeur absolue, les importances, puis les courbes ICE de `AveOccup` et de `HouseAge` avec leur moyenne. Rien à taper.

Dans tes notes : range les features par |coefficient| et par importance : lesquelles changent de rang, et pourquoi une régression linéaire et un gradient boosting peuvent-ils ne pas s'appuyer sur les mêmes features ? Pourquoi la permutation se calcule-t-elle sur la validation, et que perdrait-on à la calculer sur l'entraînement (🔁 R2) ? Sur les courbes ICE de `AveOccup`, où la dépendance partielle résume-t-elle bien les courbes, et où les cache-t-elle ? Que deviennent ces courbes au-delà de la grille, là où il y a peu de districts (piège « une dépendance partielle hors des données ») ?

In [ ]:
def linear_12(X, y):
    """A Pipeline StandardScaler -> LinearRegression, fitted on (X, y), returned."""
    raise NotImplementedError("linear_12")


def coefs_12(pipeline):
    """pd.Series of the 8 coefficients of the LinearRegression of the pipeline, indexed by the features."""
    raise NotImplementedError("coefs_12")


def boosting_12(X, y):
    """HistGradientBoostingRegressor(random_state=0), every other setting by default, fitted on (X, y), returned."""
    raise NotImplementedError("boosting_12")


def permutation_12(model, X, y):
    """pd.Series: the mean drop of R² of permutation_importance(model, X, y, n_repeats=5, random_state=0), indexed by
    the columns of X, sorted from the largest to the smallest."""
    raise NotImplementedError("permutation_12")


def pd_ice_12(model, X, feature):
    """dict {"grid": (50,), "average": (50,), "individual": (len(X), 50)} of
    partial_dependence(model, X, [feature], kind="both", grid_resolution=50)."""
    raise NotImplementedError("pd_ice_12")

In [ ]:
ROWS_12 = X_VAL.iloc[:300]                     # the districts of the curves of the check
REF_LIN_12 = make_pipeline(StandardScaler(), LinearRegression()).fit(X_TRAIN, Y_TRAIN)
ok_12 = {}
with checked("B6.12"):
    lin_12 = linear_12(X_TRAIN.copy(), Y_TRAIN.copy())
    if returned("B6.12", "linear_12", lin_12):
        steps_12 = [step for _, step in getattr(lin_12, "steps", [])]
        verdict("B6.12", isinstance(lin_12, Pipeline) and steps_12 and isinstance(steps_12[-1], LinearRegression)
                and any(isinstance(s, StandardScaler) for s in steps_12),
                "linear_12 rend un Pipeline StandardScaler → LinearRegression.",
                f"linear_12 doit rendre un Pipeline (StandardScaler, puis LinearRegression), reçu {type(lin_12).__name__} "
                f"{[type(s).__name__ for s in steps_12]}.")
        scaler_12 = next((s for s in steps_12 if isinstance(s, StandardScaler)), None)
        if scaler_12 is not None and hasattr(scaler_12, "mean_"):
            verdict("B6.12", np.allclose(scaler_12.mean_, X_TRAIN.mean().to_numpy()),
                    "le StandardScaler a appris les moyennes de X_TRAIN.",
                    "le StandardScaler n'a pas appris les moyennes de X_TRAIN : sur quelles lignes est-il ajusté ?")
        c_12 = coefs_12(lin_12)
        want_c_12 = pd.Series(REF_LIN_12[-1].coef_, index=FEATURES)
        raw_12 = LinearRegression().fit(X_TRAIN, Y_TRAIN).coef_
        if not isinstance(c_12, pd.Series) or list(c_12.index) != FEATURES:
            print("❌ Ex B6.12 : coefs_12 doit rendre une pd.Series indexée par les 8 features, dans l'ordre de FEATURES "
                  f"(reçu {type(c_12).__name__}).")
        elif np.allclose(c_12.to_numpy(), raw_12):
            print("❌ Ex B6.12 : ce sont les coefficients des features brutes : chacun dépend de l'unité de sa feature ; "
                  "le contrat les veut sur des features standardisées.")
        else:
            ok_12["coefs"] = same("B6.12", "coefficients sur features standardisées", c_12.to_numpy(),
                                  want_c_12.to_numpy(), rtol=1e-6, atol=1e-9)

with checked("B6.12"):
    gb_12 = boosting_12(X_TRAIN.copy(), Y_TRAIN.copy())
    if returned("B6.12", "boosting_12", gb_12):
        if not isinstance(gb_12, HistGradientBoostingRegressor):
            print(f"❌ Ex B6.12 : boosting_12 doit rendre un HistGradientBoostingRegressor, reçu {type(gb_12).__name__}.")
            raise StopCheck
        want_p_12 = HistGradientBoostingRegressor(random_state=0).get_params()
        diff_12 = {k: v for k, v in gb_12.get_params().items() if want_p_12.get(k) != v}
        verdict("B6.12", not diff_12, "réglages : random_state=0, le reste par défaut.",
                f"réglages différents du contrat : {diff_12}.")
        same("B6.12", "prédictions de boosting_12 sur 200 districts de validation", gb_12.predict(X_VAL.iloc[:200]),
             GBR.predict(X_VAL.iloc[:200]), rtol=1e-9, atol=1e-9)

with checked("B6.12"):
    imp_12 = permutation_12(GBR, X_VAL.copy(), Y_VAL.copy())      # the reference model: a slip above does not spread
    want_i_12 = pd.Series(permutation_importance(GBR, X_VAL, Y_VAL, n_repeats=5, random_state=0).importances_mean,
                          index=FEATURES).sort_values(ascending=False)
    if not isinstance(imp_12, pd.Series) or sorted(imp_12.index) != sorted(FEATURES):
        print("❌ Ex B6.12 : permutation_12 doit rendre une pd.Series indexée par les 8 features "
              f"(reçu {type(imp_12).__name__}).")
        raise StopCheck
    verdict("B6.12", list(imp_12.index) == list(want_i_12.index) or bool((np.diff(imp_12.to_numpy()) <= 0).all()),
            "importances triées de la plus grande à la plus petite.",
            "permutation_12 doit trier les importances de la plus grande à la plus petite.")
    if not np.allclose(imp_12[FEATURES].to_numpy(), want_i_12[FEATURES].to_numpy(), atol=1e-9):
        on_train_12 = permutation_importance(GBR, X_TRAIN, Y_TRAIN, n_repeats=5, random_state=0).importances_mean
        if np.allclose(imp_12[FEATURES].to_numpy(), on_train_12, atol=1e-9):
            print("❌ Ex B6.12 : ces importances sont celles des districts d'entraînement : la permutation mesure ce que "
                  "le modèle perd sur des données qu'il n'a pas vues (relis 206.2).")
        else:
            same("B6.12", "importances par permutation (validation, 5 répétitions, random_state=0)",
                 imp_12[FEATURES].to_numpy(), want_i_12[FEATURES].to_numpy(), atol=1e-9)
    else:
        ok_12["perm"] = True
        print("✅ Ex B6.12 : importances par permutation sur la validation, comme attendu.")

with checked("B6.12"):
    out_12 = pd_ice_12(GBR, ROWS_12.copy(), "AveOccup")
    if not isinstance(out_12, dict) or set(out_12) != {"grid", "average", "individual"}:
        print("❌ Ex B6.12 : pd_ice_12 doit rendre un dictionnaire aux trois clés 'grid', 'average', 'individual'.")
        raise StopCheck
    ref_12 = partial_dependence(GBR, ROWS_12, ["AveOccup"], kind="both", grid_resolution=50)
    same("B6.12", "grid de AveOccup (50 valeurs)", out_12["grid"], ref_12["grid_values"][0])
    same("B6.12", "courbes ICE de AveOccup (300 districts × 50 valeurs)", out_12["individual"],
         ref_12["individual"][0], rtol=1e-9, atol=1e-9)
    if same("B6.12", "dépendance partielle de AveOccup", out_12["average"], ref_12["average"][0], rtol=1e-9, atol=1e-9):
        ok_12["pd"] = True
        verdict("B6.12", np.allclose(out_12["average"], np.asarray(out_12["individual"]).mean(axis=0)),
                "la dépendance partielle est la moyenne des courbes ICE.",
                "la dépendance partielle devrait être la moyenne des courbes ICE.")

if len(ok_12) == 3:                              # the figure, once your three results are right
    fig_12, axes_12 = plt.subplots(1, 4, figsize=(17, 3.8))
    order_12 = imp_12.index
    axes_12[0].barh(order_12[::-1], c_12[order_12[::-1]].abs(), color="tab:gray")
    axes_12[0].set_title("|coefficient| (features standardisées)", fontsize=10)
    axes_12[1].barh(order_12[::-1], imp_12[order_12[::-1]], color="tab:green")
    axes_12[1].set_title("permutation : baisse du R² (validation)", fontsize=10)
    for ax_12, feat_12 in zip(axes_12[2:], ("AveOccup", "HouseAge")):
        curves_12 = pd_ice_12(GBR, ROWS_12, feat_12)
        ax_12.plot(curves_12["grid"], np.asarray(curves_12["individual"])[:80].T, color="tab:blue", alpha=0.15, lw=0.8)
        ax_12.plot(curves_12["grid"], curves_12["average"], color="black", lw=3, label="dépendance partielle")
        ax_12.set_xlabel(f"{feat_12} imposé")
        ax_12.set_ylabel("prédiction (MedHouseVal)")
        ax_12.set_title(f"80 courbes ICE de {feat_12} et leur moyenne", fontsize=10)
        ax_12.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print("coefficients standardisés :", ", ".join(f"{k} {fr(v, 2)}" for k, v in c_12.items()))
    print("importances par permutation :", ", ".join(f"{k} {fr(v, 3)}" for k, v in imp_12.items()))

### Ex B6.13 — Prédire la forme des courbes ICE du revenu médian et de la latitude 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que les courbes ICE d'un gradient boosting montrent de deux features, avant de les tracer.  
**Prérequis :** Ex B6.12 · fiche §206.2 (« La dépendance partielle », « Les courbes ICE ») · **Fil rouge :** California · **Parcours :** C

L'expérience trace, pour le gradient boosting de référence `GBR`, les courbes ICE de `MedInc` (le revenu médian du district, en dizaines de milliers de dollars) et de `Latitude` (de ≈ 33° au sud, vers San Diego, à ≈ 39° au nord de la grille) sur 500 districts de validation, avec leur moyenne, la dépendance partielle ; chaque grille a 50 valeurs, du 5ᵉ au 95ᵉ centile de la feature. La prédiction est en `MedHouseVal`, en centaines de milliers de dollars. **Avant d'exécuter**, écris tes quatre prédictions :
- a) `medinc_rise_13` : de combien la dépendance partielle de `MedInc` monte-t-elle du début à la fin de sa grille ? `"A"` moins de 0,5 ; `"B"` de 0,5 à 1,5 ; `"C"` de 1,5 à 3 ; `"D"` plus de 3 ;
- b) `lat_trend_13` : du sud au nord, la dépendance partielle de `Latitude` : `"A"` monte (de plus de 0,3) ; `"B"` reste à peu près plate (entre −0,3 et +0,3) ; `"C"` descend (de plus de 0,3) ;
- c) `lat_up_13` : quelle part des 500 courbes ICE de `Latitude` **finissent plus haut** qu'elles ne commencent ? `"A"` moins de 5 % ; `"B"` de 5 à 30 % ; `"C"` de 30 à 70 % ; `"D"` plus de 70 % ;
- d) `spread_13` : les courbes ICE de `Latitude` varient-elles **plus** d'un district à l'autre que celles de `MedInc` (un écart-type plus grand de leur variation, fin moins début) ? `True` ou `False`.

La cellule d'expérience n'affiche rien tant que tes quatre prédictions ne sont pas écrites ; elle les vérifie à la fin. Rien n'y dépend de ton code.

Dans tes notes, avant d'exécuter : ce que tu sais de la Californie (où sont les villes, la côte, les grandes régions) et ce que tu attends d'un modèle qui voit la latitude **et** la longitude.

In [ ]:
ROWS_13 = X_VAL.iloc[:500]                        # the 500 validation districts of the curves


def curves_13(feature):
    """(grid, average, individual) of the reference gradient boosting GBR for `feature` on ROWS_13: a grid of 50 values
    between the 5th and the 95th percentiles of the feature (written here without your code)."""
    result = partial_dependence(GBR, ROWS_13, [feature], kind="both", grid_resolution=50)
    return result["grid_values"][0], result["average"][0], result["individual"][0]

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
medinc_rise_13 = ...   # a) by how much does the partial dependence of MedInc rise from the start to the end of its
                       #    grid? "A" less than 0.5, "B" 0.5 to 1.5, "C" 1.5 to 3, "D" more than 3 (units of MedHouseVal)
lat_trend_13 = ...     # b) from the south to the north of its grid, the partial dependence of Latitude: "A" rises
                       #    (by more than 0.3), "B" stays about flat (between -0.3 and +0.3), "C" falls (by more than 0.3)
lat_up_13 = ...        # c) which share of the 500 ICE curves of Latitude END HIGHER than they start? "A" less than 5 %,
                       #    "B" 5 to 30 %, "C" 30 to 70 %, "D" more than 70 %
spread_13 = ...        # d) do the ICE curves of Latitude vary more from one district to another than those of MedInc
                       #    (a larger standard deviation of their change, end minus start)? True or False

In [ ]:
if any(answer is ... or answer is None for answer in [medinc_rise_13, lat_trend_13, lat_up_13, spread_13]):
    print("⏳ Ex B6.13 : écris d'abord tes quatre prédictions dans la cellule précédente.")
else:
    fig_13, axes_13 = plt.subplots(1, 2, figsize=(12, 4))
    seen_13 = {}
    for ax_13, feature_13 in zip(axes_13, ("MedInc", "Latitude")):
        grid_13, avg_13, ind_13 = curves_13(feature_13)
        ax_13.plot(grid_13, ind_13[:100].T, color="tab:blue", alpha=0.15, lw=0.8)
        ax_13.plot(grid_13, avg_13, color="black", lw=3, label="dépendance partielle")
        ax_13.set_xlabel(f"{feature_13} imposé (du 5e au 95e centile)")
        ax_13.set_ylabel("prédiction (MedHouseVal)")
        ax_13.set_title(f"100 courbes ICE de {feature_13} (sur 500) et leur moyenne", fontsize=10)
        ax_13.legend(fontsize=8)
        change_13 = ind_13[:, -1] - ind_13[:, 0]
        seen_13[feature_13] = {"rise": float(avg_13[-1] - avg_13[0]), "up": float(np.mean(change_13 > 0)),
                               "spread": float(np.std(change_13))}
    plt.tight_layout()
    plt.show()
    for feature_13, s_13 in seen_13.items():
        print(f"{feature_13:>8} : variation de la dépendance partielle (fin − début) {fr(s_13['rise'], 2)} ; "
              f"courbes ICE qui finissent plus haut {pct(s_13['up'])} ; écart-type de la variation des ICE {fr(s_13['spread'], 2)}")
    wb.check("B6.13a", medinc_rise_13)
    wb.check("B6.13b", lat_trend_13)
    wb.check("B6.13c", lat_up_13)
    wb.check("B6.13d", spread_13)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Que te disent les courbes ICE de la latitude que la dépendance partielle seule ne dit pas, et avec quelle autre feature la latitude interagit-elle sans doute ? Imposer une latitude à un district sans toucher à sa longitude : quelles combinaisons crée-t-on, et ces districts existent-ils (piège « une dépendance partielle hors des données ») ? Une courbe ICE dit-elle ce que **ferait** un déménagement du district vers le nord ?

### Ex B6.14 — shapley_values : Shapley exact par énumération des coalitions 🔨 ★★★ ⏱️ 45 min
**Objectif :** écrire le calcul exact des valeurs de Shapley interventionnelles, que shap approche.  
**Prérequis :** ✏️ B6.1 · ∂ B6.2 · fiche §206.3 (« Les features comme joueurs », « Un exemple à deux features », « Combien ça coûte ») · **Fil rouge :** California · **mylearn :** `explain.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/explain.py` (créé par `python tools/start_chapter.py B6`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb6_explain.py -q` les lance tous dans un terminal.

**Contrat** de `shapley_values(f, x, background, max_features=12)`, qui rend le couple `(phi, base_value)` :
- `f` ne reçoit que des tableaux 2-D de forme `(n, M)`, jamais une ligne seule, et rend une sortie de forme `(n,)`, par exemple `model.predict(X)` ou `model.predict_proba(X)[:, 1]` ; elle peut être appelée plusieurs fois ;
- $v(S)$ est la **moyenne de $f$** sur les $B$ lignes du fond dont les colonnes de $S$ prennent les valeurs de `x` ($f$ d'abord, la moyenne ensuite) ; **toutes** les lignes du fond comptent, sans sous-échantillonnage (contrairement au `max_samples=100` par défaut de shap) ;
- `phi` est un tableau de forme `(M,)` ; `base_value` vaut $v(\varnothing)$, la moyenne de $f$ sur le fond, un `float` Python ; l'efficacité `phi.sum() + base_value == f(x)` tient aux arrondis près ;
- `ValueError` si $M >$ `max_features` ($M =$ `max_features` est permis), si `x` n'est pas 1-D, ou si `background` n'est pas 2-D à $M$ colonnes ; `x` et `background` peuvent être des listes et ne sont jamais modifiés ;
- **coût** : la docstring prescrit l'énumération des $2^M$ coalitions, chaque $v(S)$ calculé **une seule fois** (un appel à `f` sur les $B$ lignes) ; les tests comptent les lignes passées à `f` et arrêtent ta fonction au-delà de `max(2·M·2**M·B, 500 000)` lignes, avec un message qui le dit. Ce budget laisse passer un $v(S)$ recalculé pour chaque feature, ou `f` appelée ligne par ligne ; il arrête une énumération des $M!$ ordres d'arrivée sans mémoire des $v(S)$ (40 320 ordres à $M = 8$), qui prendrait des minutes.

La première cellule de vérification regarde d'abord ce que reçoit `f`, puis essaie ta fonction sur les deux exemples de la docstring, sur un modèle linéaire à 5 features et un fond aléatoire de 20 lignes ($\phi_i = w_i(x_i - \bar b_i)$, ∂ B6.2), sur le gradient boosting de référence de California (8 features, fond de 50 lignes : efficacité, valeur de base, comparaison à `shap.explainers.Exact`, durée affichée), et sur quatre gardes. La seconde lance les 33 tests de l'exercice (`-k "test_shapley_values_"`).

Dans tes notes : combien de lignes ta fonction passe-t-elle à `f` pour $M = 8$ et $B = 50$, et combien en passerait une version qui recalcule chaque $v(S)$ pour chaque feature ? Pourquoi `f` doit-elle recevoir des tableaux 2-D (pense au coût d'un appel à `predict`) ? Que deviendrait la durée affichée avec 12 features (🧮 B6.7) ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B6")   # reload your saved file
with checked("B6.14"):
    E = mylearn.explain
    rng_14 = np.random.default_rng(14)                 # a linear model on a random background: w_i (x_i - mean_i)
    w5_14, x5_14, bg5_14 = rng_14.normal(size=5), rng_14.normal(size=5), rng_14.normal(size=(20, 5))
    shapes_14 = []

    def spy_14(X):                                     # first: what does f receive?
        shapes_14.append(np.shape(X))
        return np.asarray(X, dtype=float) @ w5_14

    E.shapley_values(spy_14, x5_14, bg5_14)
    bad_14 = sorted({s for s in shapes_14 if len(s) != 2 or s[1] != 5})
    verdict("B6.14", not bad_14, "f ne reçoit que des tableaux 2-D de 5 colonnes.",
            f"f a reçu les formes {bad_14[:3]} : appelle-la sur des tableaux (n, M), jamais sur une ligne seule.")
    rows_14 = sum(s[0] for s in shapes_14 if len(s) == 2)
    print(f"ℹ️ f a été appelée {len(shapes_14)} fois, sur {rows_14} lignes en tout (2**5 coalitions × 20 lignes = "
          f"{2 ** 5 * 20}).")
    w_14 = np.array([2.0, -1.0, 0.5])
    bg_14 = np.array([[0.0, 0.0, 0.0], [2.0, 2.0, 2.0]])
    phi_14, base_14 = pair("B6.14", "shapley_values", E.shapley_values(lambda X: X @ w_14, [3.0, 1.0, 1.0], bg_14.copy()),
                           names=("phi", "base_value"), floats=("base_value",))
    same("B6.14", "phi du modèle linéaire de la docstring", phi_14, [4.0, 0.0, 0.0])
    same("B6.14", "base_value du modèle linéaire de la docstring", base_14, 1.5)
    same("B6.14", "le produit x0 x1 de la docstring : l'interaction partagée",
         E.shapley_values(lambda X: X[:, 0] * X[:, 1], [1.0, 1.0], [[0.0, 0.0]])[0], [0.5, 0.5])

    same("B6.14", "modèle linéaire à 5 features, fond de 20 lignes : w_i (x_i − moyenne du fond)",
         E.shapley_values(lambda X: X @ w5_14 + 1.0, x5_14, bg5_14)[0], w5_14 * (x5_14 - bg5_14.mean(axis=0)),
         rtol=1e-8, atol=1e-10)

    X0_14, BG_14 = X_VAL.iloc[0].to_numpy(), X_TRAIN.iloc[:50].to_numpy()
    copies_14 = (X0_14.copy(), BG_14.copy())
    start_14 = time.perf_counter()
    phi_c_14, base_c_14 = pair("B6.14", "shapley_values", E.shapley_values(GBR.predict, X0_14, BG_14),
                               names=("phi", "base_value"), floats=("base_value",))
    seconds_14 = time.perf_counter() - start_14
    same("B6.14", "California, efficacité : somme des phi + base_value = f(x)", float(np.sum(phi_c_14)) + base_c_14,
         float(GBR.predict(X0_14[None])[0]), rtol=1e-9, atol=1e-9)
    same("B6.14", "California : base_value = moyenne de f sur les 50 lignes du fond", base_c_14,
         float(GBR.predict(BG_14).mean()), rtol=1e-9, atol=1e-9)
    if shap is not None:
        exact_14 = shap.explainers.Exact(GBR.predict, shap.maskers.Independent(BG_14, max_samples=len(BG_14)))(X0_14[None])
        same("B6.14", "California : phi contre shap.explainers.Exact (fond complet)", phi_c_14, exact_14.values[0],
             rtol=1e-7, atol=1e-9)
    print(f"ℹ️ California, 8 features, fond de 50 lignes (256 coalitions × 50 = 12 800 lignes) : {fr(seconds_14, 2)} s.")
    verdict("B6.14", np.array_equal(X0_14, copies_14[0]) and np.array_equal(BG_14, copies_14[1]),
            "x et le fond n'ont pas changé.", "shapley_values a modifié x ou le fond reçus.")

    try:
        E.shapley_values(lambda X: X.sum(axis=1), np.ones(3), np.zeros((2, 3)), max_features=3)
        print("✅ Ex B6.14 : M = max_features est permis.")
    except ValueError as exc_14:
        print(f"❌ Ex B6.14 : M = max_features doit être permis (seul M > max_features lève) : reçu ValueError ({exc_14}).")
    refuses("B6.14", "M = 4 > max_features = 3", lambda: E.shapley_values(lambda X: X.sum(axis=1), np.ones(4),
                                                                        np.zeros((2, 4)), max_features=3))
    refuses("B6.14", "x en 2-D", lambda: E.shapley_values(lambda X: X.sum(axis=1), np.ones((1, 3)), np.zeros((2, 3))))
    refuses("B6.14", "un fond de 2 colonnes pour x de 3", lambda: E.shapley_values(lambda X: X.sum(axis=1), np.ones(3),
                                                                                   np.zeros((2, 2))))

In [ ]:
if written(lambda: mylearn.explain.shapley_values(lambda X: X.sum(axis=1), [0.0], [[0.0]])):
    run_pytest_group(TEST_EXPLAIN, "test_shapley_values_")   # 33 tests
else:
    print("⏳ Ex B6.14 : les tests attendent ta fonction shapley_values.")

### Ex B6.15 — SHAP sur un gradient boosting California : TreeExplainer, waterfall, beeswarm, dépendance 📦 ★★ ⏱️ 30 min
**Objectif :** expliquer un gradient boosting avec TreeSHAP, vérifier qu'il calcule le même Shapley que ta fonction, et lire les trois graphiques de shap.  
**Prérequis :** Ex B6.14 · ch. 14 · fiche §206.4 (« L'API de shap », « Les trois graphiques à connaître ») · **Fil rouge :** California · **Parcours :** R, C

**Contrat** des deux fonctions de la cellule suivante :
- `fit_15(X, y)` rend un `HistGradientBoostingRegressor(max_depth=3, random_state=0)` (des arbres peu profonds, les autres réglages par défaut), ajusté sur `(X, y)` ;
- `tree_shap_15(model, background, rows)` rend le `shap.Explanation` de `shap.TreeExplainer(model, data=background, feature_perturbation="interventional")` appliqué au DataFrame `rows` : `.values` de forme `(len(rows), 8)`, `.base_values`, `.feature_names`.

La cellule donnée fixe `BG_15`, le fond (les 50 premiers districts d'entraînement), et `ROWS_15`, les 1 000 premiers districts de validation à expliquer. La cellule de vérification ajuste ton modèle sur `X_TRAIN`, l'explique, contrôle l'efficacité sur les 1 000 districts, puis vérifie quatre grandeurs que tes fonctions calculent (rien à taper, aucune à arrondir) : a) la valeur de base ; b) la valeur SHAP de `MedInc` pour le premier district ; c) la moyenne des $\lvert\phi_{\text{MedInc}}\rvert$ sur les 1 000 districts ; d) si ta `shapley_values` (B6.14), avec le même fond de 50 lignes, redonne les valeurs de TreeSHAP sur le premier district à $10^{-6}$ près (`True` ou `False`). La cellule suivante trace le waterfall du premier district, le beeswarm et la dépendance de `MedInc`.

> ⚠️ **Piège classique — TreeSHAP et les données en float64** — `TreeExplainer` parcourt les arbres en **float32**. Un `HistGradientBoosting` pose certains seuils exactement sur des valeurs répétées des données (la latitude et la longitude de California en ont beaucoup) : une valeur égale à un seuil peut alors passer de l'autre côté, et TreeSHAP perd l'efficacité (des écarts jusqu'à 0,5 sur les données d'origine de scikit-learn, mesurés avec shap 0.54.1). C'est pourquoi `wbx.california()` arrondit les features en float32 : TreeSHAP interventionnel y redonne le Shapley exact à $10^{-8}$ près. Sur tes propres données, vérifie toujours l'efficacité (shap le fait : `check_additivity`).

Dans tes notes : pourquoi la valeur de base a) n'est-elle ni la moyenne des cibles d'entraînement, ni la prédiction moyenne des 1 000 districts expliqués ? Sur le waterfall, quelles features font monter et descendre la prédiction du premier district, et de combien ? Sur le beeswarm, dans quel sens joue un grand revenu, et quelles features ont une grande dispersion ? Sur la dépendance de `MedInc`, qu'indique la dispersion verticale à revenu égal ? Combien de temps a pris TreeSHAP sur 1 000 districts, et combien prendrait ta `shapley_values` (sa durée pour un district est affichée en B6.14) ?

In [ ]:
BG_15 = X_TRAIN.iloc[:50]          # the background: the first 50 training districts
ROWS_15 = X_VAL.iloc[:1000]        # the 1 000 validation districts to explain
MEDINC = FEATURES.index("MedInc")  # the column of MedInc (0)

In [ ]:
def fit_15(X, y):
    """HistGradientBoostingRegressor(max_depth=3, random_state=0), every other setting by default, fitted on (X, y)."""
    raise NotImplementedError("fit_15")


def tree_shap_15(model, background, rows):
    """The shap.Explanation of shap.TreeExplainer(model, data=background, feature_perturbation="interventional")
    on the DataFrame `rows`: .values of shape (len(rows), 8), .base_values, .feature_names."""
    raise NotImplementedError("tree_shap_15")

In [ ]:
SV_15 = MODEL_15 = None
with checked("B6.15"):
    if shap is None:
        print("⏳ Ex B6.15 : cet exercice a besoin de shap (wb.ensure('shap')).")
        raise StopCheck
    model_15 = fit_15(X_TRAIN.copy(), Y_TRAIN.copy())
    if not returned("B6.15", "fit_15", model_15):
        raise StopCheck
    if not isinstance(model_15, HistGradientBoostingRegressor):
        print(f"❌ Ex B6.15 : fit_15 doit rendre un HistGradientBoostingRegressor, reçu {type(model_15).__name__}.")
        raise StopCheck
    want_p_15 = HistGradientBoostingRegressor(max_depth=3, random_state=0).get_params()
    diff_15 = {k: v for k, v in model_15.get_params().items() if want_p_15.get(k) != v}
    verdict("B6.15", not diff_15, "réglages : max_depth=3, random_state=0, le reste par défaut.",
            f"réglages différents du contrat : {diff_15}.")
    ref_15 = HistGradientBoostingRegressor(max_depth=3, random_state=0).fit(X_TRAIN, Y_TRAIN)
    if not same("B6.15", "prédictions de fit_15 sur 200 districts de validation", model_15.predict(X_VAL.iloc[:200]),
                ref_15.predict(X_VAL.iloc[:200]), rtol=1e-9, atol=1e-9):
        raise StopCheck
    MODEL_15 = model_15
    start_15 = time.perf_counter()
    sv_15 = tree_shap_15(model_15, BG_15.copy(), ROWS_15.copy())
    print(f"ℹ️ tree_shap_15 sur les 1 000 districts : {fr(time.perf_counter() - start_15, 1)} s.")
    if not returned("B6.15", "tree_shap_15", sv_15):
        raise StopCheck
    if isinstance(sv_15, np.ndarray):
        print("❌ Ex B6.15 : tree_shap_15 rend un tableau NumPy : le contrat veut le shap.Explanation de explainer(rows) "
              "(avec .base_values et .feature_names), pas explainer.shap_values(rows).")
        raise StopCheck
    if np.shape(getattr(sv_15, "values", None)) != (len(ROWS_15), len(FEATURES)):
        print("❌ Ex B6.15 : tree_shap_15 doit rendre un shap.Explanation dont .values a la forme (1000, 8), reçu "
              f"{type(sv_15).__name__} de forme {np.shape(getattr(sv_15, 'values', None))}.")
        raise StopCheck
    if not getattr(sv_15, "feature_names", None):
        sv_15.feature_names = list(FEATURES)               # rows given as an array: the names for the plots
    SV_15 = sv_15
    gap_eff_15 = float(np.abs(sv_15.values.sum(axis=1) + np.ravel(sv_15.base_values) - model_15.predict(ROWS_15)).max())
    verdict("B6.15", gap_eff_15 < 1e-6, f"efficacité sur les 1 000 districts : écart maximal {gap_eff_15:.1e}.",
            f"efficacité : somme des valeurs SHAP + valeur de base s'écarte de la prédiction de {gap_eff_15:.2g} : "
            "l'explication porte-t-elle sur ce modèle et sur ces lignes ?")
    wb.check("B6.15a", float(np.ravel(sv_15.base_values)[0]), computed=True)
    wb.check("B6.15b", float(sv_15.values[0, MEDINC]), computed=True)
    wb.check("B6.15c", float(np.mean(np.abs(sv_15.values[:, MEDINC]))), computed=True)

with checked("B6.15"):
    if SV_15 is None:
        raise StopCheck
    if not written(lambda: mylearn.explain.shapley_values(lambda X: X.sum(axis=1), [0.0], [[0.0]])):
        print("⏳ Ex B6.15 : la comparaison attend ta shapley_values (Ex B6.14).")
        raise StopCheck
    x0_15 = ROWS_15.iloc[0].to_numpy()
    phi_15, base_15 = pair("B6.15", "shapley_values", mylearn.explain.shapley_values(MODEL_15.predict, x0_15,
                                                                                     BG_15.to_numpy()),
                           names=("phi", "base_value"), floats=("base_value",))
    gap_15 = max(float(np.abs(np.asarray(phi_15, dtype=float) - SV_15.values[0]).max()),
                 abs(base_15 - float(np.ravel(SV_15.base_values)[0])))
    print(f"ℹ️ écart maximal entre TreeExplainer et ta shapley_values sur le premier district : {gap_15:.1e}")
    wb.check("B6.15d", bool(gap_15 <= 1e-6))

In [ ]:
if SV_15 is not None:                       # the three plots of 206.4, on your explanation
    shap.plots.waterfall(SV_15[0], show=False)
    plt.title("waterfall du premier district", fontsize=10)
    plt.show()
    shap.plots.beeswarm(SV_15, show=False)
    plt.title("beeswarm des 1 000 districts", fontsize=10)
    plt.show()
    shap.plots.scatter(SV_15[:, MEDINC], color=SV_15, show=False)
    plt.title("dépendance de MedInc (couleur : la feature qui interagit le plus)", fontsize=10)
    plt.show()
else:
    print("⏳ Ex B6.15 : les graphiques attendent ton explication (cellule précédente).")

### Ex B6.16 — Le fond change tout : trois fonds différents, puis une feature dupliquée 🔬 ★★★ ⏱️ 40 min
**Objectif :** mesurer ce que le choix du fond et une feature dupliquée font aux valeurs de Shapley d'un gradient boosting.  
**Prérequis :** Ex B6.15 · fiche §206.3 (« Le fond fait partie de la question ») et §206.4 (« Features corrélées : ce que SHAP répartit ») · 🧠 Q5, Q6 · **Fil rouge :** California · **Parcours :** C

Deux expériences sur le gradient boosting de référence `GBR`. **Ne présume pas l'issue : mesure-la.**

**1. Trois fonds.** La cellule donnée choisit `X_16`, le district de validation le plus proche du 75ᵉ centile du revenu, et trois fonds (`BACKGROUNDS_16`) : `"population"`, 50 districts d'entraînement tirés au hasard ; `"riches"`, 50 districts du décile de revenu le plus haut ; `"ligne type"`, **une** ligne, la médiane de chaque feature. Contrat de `explain_16(model, x, background)` : il rend le couple `(phi, base)`, les valeurs de Shapley interventionnelles de `model.predict` au district `x` (une `pd.Series` des 8 features) avec le fond `background` (un DataFrame), un tableau `(8,)`, et la valeur de base, un `float` ; ta `shapley_values` ou `shap.TreeExplainer` (les mêmes nombres, B6.15). La cellule de vérification compare tes valeurs à TreeSHAP pour les trois fonds, puis affiche le tableau des valeurs de base et des $\phi$.

**2. Une feature dupliquée.** `add_copy_16(X, noise, rng)` (donnée) ajoute à un DataFrame, juste après `MedInc`, la colonne `"MedInc_copy"` = `MedInc` + `noise` × (écart-type de `MedInc`) × un bruit gaussien tiré par `rng`. Contrat de `copy_experiment_16(noise, seed, n_rows=200)` :
- `rng = np.random.default_rng(seed)`, puis `A = add_copy_16(X_TRAIN, noise, rng)` et `V = add_copy_16(X_VAL, noise, rng)`, dans cet ordre, avec le même `rng` ;
- un `HistGradientBoostingRegressor(random_state=seed)` ajusté sur `(A, Y_TRAIN)` ;
- `shap.TreeExplainer` interventionnel, de fond les 50 premières lignes de `A`, sur les `n_rows` premières lignes de `V` ;
- il rend `{"MedInc": moyenne des |φ_MedInc|, "MedInc_copy": moyenne des |φ_copie|, "sum": moyenne des |φ_MedInc + φ_copie|}` (des floats).

La cellule d'expérience essaie d'abord ta fonction sur un petit cas (bruit 0,01, graine 0, 50 districts), puis la lance pour trois niveaux de bruit (0, 0,01 et 0,1 écart-type) et trois graines (≈ 10 s), à côté de la moyenne des $\lvert\phi_{\text{MedInc}}\rvert$ du modèle sans copie.

Dans tes notes : avec quel fond la valeur SHAP de `MedInc` change-t-elle de signe, et pourquoi ? Que change le fond « ligne type » par rapport à « population » ? Quelle phrase écrirais-tu sous un waterfall pour dire quel fond a servi ? Pour la copie : que reçoit-elle quand elle est exacte (bruit 0), et pourquoi ? Avec un bruit de 0,01, le partage entre `MedInc` et sa copie est-il stable d'une graine à l'autre ? Et la somme ? Que rapporterais-tu à un métier qui demande « quelle est l'importance du revenu ? » quand deux colonnes le portent ?

In [ ]:
# the district to explain: the validation district closest to the 75th percentile of MedInc
X_16 = X_VAL.iloc[int(np.argmin(np.abs(X_VAL["MedInc"].to_numpy() - X_VAL["MedInc"].quantile(0.75))))]
RICH_16 = X_TRAIN[X_TRAIN["MedInc"] >= X_TRAIN["MedInc"].quantile(0.9)]          # the top decile of income
BACKGROUNDS_16 = {"population": X_TRAIN.sample(50, random_state=0),             # 50 districts drawn at random
                  "riches": RICH_16.sample(50, random_state=0),                  # 50 districts of the top decile
                  "ligne type": X_TRAIN.median().to_frame().T.astype("float32").astype("float64")}   # one row: medians
SD_MEDINC_16 = float(X_TRAIN["MedInc"].std())
NOISES_16, SEEDS_16 = (0.0, 0.01, 0.1), (0, 1, 2)


def add_copy_16(X, noise, rng):
    """A copy of the DataFrame X with the column "MedInc_copy" = MedInc + noise * SD_MEDINC_16 * N(0, 1), drawn by
    rng.standard_normal(len(X)), placed right after MedInc and rounded to float32 like the other features (B6.15)."""
    out = X.copy()
    copy = X["MedInc"].to_numpy() + noise * SD_MEDINC_16 * rng.standard_normal(len(X))
    out.insert(1, "MedInc_copy", copy.astype("float32").astype("float64"))
    return out


print(f"district expliqué : MedInc {fr(X_16['MedInc'], 2)}, Latitude {fr(X_16['Latitude'], 2)}, prédiction "
      f"{fr(GBR.predict(X_16.to_frame().T)[0], 2)} ; fonds : " + ", ".join(f"{k} ({len(v)} ligne{'s' * (len(v) > 1)})"
                                                                            for k, v in BACKGROUNDS_16.items()))

In [ ]:
def explain_16(model, x, background):
    """(phi, base): the interventional Shapley values of model.predict at the district x (a pd.Series of the 8
    features) with the DataFrame `background`, as an array (8,), and the base value, a float."""
    raise NotImplementedError("explain_16")


def copy_experiment_16(noise, seed, n_rows=200):
    """The duplicated feature: rng = np.random.default_rng(seed); A = add_copy_16(X_TRAIN, noise, rng), then
    V = add_copy_16(X_VAL, noise, rng) (the same rng, in this order); HistGradientBoostingRegressor(random_state=seed)
    fitted on (A, Y_TRAIN); shap.TreeExplainer (interventional, background: the first 50 rows of A) on the first
    n_rows rows of V. Returns {"MedInc": mean |phi_MedInc|, "MedInc_copy": mean |phi_copy|,
    "sum": mean |phi_MedInc + phi_copy|} over these rows (floats)."""
    raise NotImplementedError("copy_experiment_16")

In [ ]:
TABLE_16 = {}
with checked("B6.16"):
    if shap is None:
        print("⏳ Ex B6.16 : cet exercice a besoin de shap (wb.ensure('shap')).")
        raise StopCheck
    for name_16, bg_16 in BACKGROUNDS_16.items():
        got_16 = explain_16(GBR, X_16.copy(), bg_16.copy())
        phi_16, base_16 = pair("B6.16", "explain_16", got_16, names=("phi", "base"), floats=("base",))
        want_16 = shap.TreeExplainer(GBR, data=bg_16, feature_perturbation="interventional")(X_16.to_frame().T)
        ok_16 = same("B6.16", f"fond « {name_16} » : phi", phi_16, want_16.values[0], rtol=1e-6, atol=1e-6)
        ok_16 &= same("B6.16", f"fond « {name_16} » : valeur de base", base_16, float(np.ravel(want_16.base_values)[0]),
                      rtol=1e-6, atol=1e-6)
        if ok_16:
            TABLE_16[name_16] = (base_16, np.asarray(phi_16, dtype=float))
if len(TABLE_16) == len(BACKGROUNDS_16):
    f_16 = float(GBR.predict(X_16.to_frame().T)[0])
    print(f"\nprédiction du district : {fr(f_16, 3)}")
    print(f"{'fond':>12} | {'base':>6} | " + " | ".join(f"{name[:9]:>9}" for name in FEATURES))
    for name_16, (base_16, phi_16) in TABLE_16.items():
        print(f"{name_16:>12} | {fr(base_16, 3):>6} | " + " | ".join(f"{fr(v, 3):>9}" for v in phi_16))

In [ ]:
COPIES_16 = {}
with checked("B6.16"):
    if shap is None:
        raise StopCheck
    got_16 = copy_experiment_16(0.01, 0, n_rows=50)      # a small first run, checked before the whole grid
    if not isinstance(got_16, dict) or set(got_16) != {"MedInc", "MedInc_copy", "sum"}:
        print("❌ Ex B6.16 : copy_experiment_16 doit rendre un dictionnaire aux clés 'MedInc', 'MedInc_copy', 'sum'.")
        raise StopCheck
    rng_16 = np.random.default_rng(0)
    A_16, V_16 = add_copy_16(X_TRAIN, 0.01, rng_16), add_copy_16(X_VAL, 0.01, rng_16)
    m_16 = HistGradientBoostingRegressor(random_state=0).fit(A_16, Y_TRAIN)
    v_16 = shap.TreeExplainer(m_16, data=A_16.iloc[:50], feature_perturbation="interventional")(V_16.iloc[:50]).values
    want_16 = [np.mean(np.abs(v_16[:, 0])), np.mean(np.abs(v_16[:, 1])), np.mean(np.abs(v_16[:, 0] + v_16[:, 1]))]
    if not same("B6.16", "copy_experiment_16(0.01, 0, n_rows=50)", [got_16[k] for k in ("MedInc", "MedInc_copy", "sum")],
                want_16, rtol=1e-6, atol=1e-6):
        raise StopCheck
    start_16 = time.perf_counter()
    for noise_16 in NOISES_16:
        for seed_16 in SEEDS_16:
            COPIES_16[noise_16, seed_16] = copy_experiment_16(noise_16, seed_16)
    print(f"ℹ️ {len(COPIES_16)} entraînements et explications en {time.perf_counter() - start_16:.0f} s.\n")
    alone_16 = shap.TreeExplainer(GBR, data=X_TRAIN.iloc[:50], feature_perturbation="interventional")(X_VAL.iloc[:200])
    print(f"sans copie : moyenne de |phi_MedInc| sur les 200 districts = {fr(np.mean(np.abs(alone_16.values[:, 0])), 3)}")
    for noise_16 in NOISES_16:
        cells_16 = [COPIES_16[noise_16, s] for s in SEEDS_16]
        print(f"bruit {fr(noise_16, 2)} écart-type : " + " ; ".join(
            f"graine {s} : MedInc {fr(c['MedInc'], 3)}, copie {fr(c['MedInc_copy'], 3)}, somme {fr(c['sum'], 3)}"
            for s, c in zip(SEEDS_16, cells_16)))

### Ex B6.17 — Penguins : SHAP par classe pour une régression logistique et une forêt aléatoire 📦 ★★ ⏱️ 25 min
**Objectif :** expliquer un classifieur à trois classes, une sortie à la fois, en logit puis en probabilité.  
**Prérequis :** Ex B6.15 · ∂ B6.2 · fiche §206.4 (« L'API de shap ») · ch. 13 et 14 · **Fil rouge :** Penguins · **Parcours :** C

Pourquoi ce manchot est-il un Gentoo ? La cellule donnée charge le découpage figé de Penguins (`wbx.penguins()` : 249 manchots d'entraînement, 84 de test, les 4 mesures en colonnes, l'espèce en label) et `GENTOO_17`, le premier Gentoo du test ; au ch. 0A tu as découvert ce dataset, au ch. 13 tu l'as classé. Un classifieur multiclasse a **une sortie par classe** : on explique une sortie à la fois, la probabilité de la classe $k$ ou son logit, et ces deux sorties ne donnent pas les mêmes valeurs (206.4). **Contrat** :
- `fit_logistic_17(X, y)` rend `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` ajusté ; `fit_forest_17(X, y)`, `RandomForestClassifier(n_estimators=200, random_state=0)` ajusté ;
- `class_output_17(model, k, kind="proba")` rend la fonction `f(X) -> (n,)` d'**une** sortie : la probabilité de la classe d'indice `k` (`predict_proba`) pour `kind="proba"`, son logit (`decision_function`) pour `kind="logit"` ;
- `shap_by_class_17(model, x, background, kind="proba")` rend le couple `(phi, base)` : `phi` de forme `(3, 4)`, une ligne par classe dans l'ordre de `model.classes_` (`SPECIES`), calculée par ta `shapley_values` sur la sortie de cette classe, et `base` de forme `(3,)`.

La cellule de vérification compare tes deux modèles à scikit-learn, puis explique `GENTOO_17` avec les modèles de référence et tout l'entraînement pour fond : en logit pour la régression logistique, elle compare tes valeurs au cas linéaire ($\phi_{ki} = w_{ki}(x_i - \bar b_i)$ dans les unités des mesures, ∂ B6.2) ; en probabilité, elle vérifie l'efficacité de chaque classe et les valeurs de base ; pour la forêt, elle compare tes valeurs à `shap.TreeExplainer` interventionnel, avec tout le fond (`shap.maskers.Independent(XP_TRAIN, max_samples=249)` : sinon il n'en garde que 100), à $2 \cdot 10^{-3}$ près, parce que sur une forêt TreeSHAP s'écarte lui-même de `predict_proba` de quelques $10^{-4}$. Puis elle trace les trois explications et affiche, pour chacune, la somme sur les trois classes de chaque feature.

Dans tes notes : quelles mesures font du manchot un Gentoo selon chaque explication ? Les logits et les probabilités classent-ils les mesures dans le même ordre ? Pourquoi les sommes sur les classes affichées valent-elles 0 pour les probabilités (pense à l'additivité et à ce que vaut la somme des trois probabilités), et que valent-elles pour les logits ? La forêt et la régression logistique s'appuient-elles sur les mêmes mesures ?

In [ ]:
PENG = wbx.penguins()                               # frozen split of Penguins: 249 penguins, 84 for the test
XP_TRAIN, XP_TEST = PENG.X_train.to_numpy(), PENG.X_test.to_numpy()   # the 4 measures, as arrays (columns PENG.features)
YP_TRAIN, YP_TEST = PENG.y_train.to_numpy(), PENG.y_test.to_numpy()   # the species, strings
SPECIES = PENG.classes                               # ['Adelie', 'Chinstrap', 'Gentoo']: the order of predict_proba
GENTOO_17 = XP_TEST[np.flatnonzero(YP_TEST == "Gentoo")[0]]           # the penguin to explain: the first Gentoo
print("le manchot expliqué :", ", ".join(f"{name} {fr(v, 1)}" for name, v in zip(PENG.features, GENTOO_17)))

In [ ]:
def fit_logistic_17(X, y):
    """make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), fitted on (X, y), returned."""
    raise NotImplementedError("fit_logistic_17")


def fit_forest_17(X, y):
    """RandomForestClassifier(n_estimators=200, random_state=0), fitted on (X, y), returned."""
    raise NotImplementedError("fit_forest_17")


def class_output_17(model, k, kind="proba"):
    """The function f(X) -> (n,) of ONE output of a classifier: the probability of the class of index k
    (predict_proba) for kind="proba", its logit (decision_function) for kind="logit"."""
    raise NotImplementedError("class_output_17")


def shap_by_class_17(model, x, background, kind="proba"):
    """(phi, base): phi of shape (3, 4), the Shapley values of the 4 measures for each class (a row per class, in the
    order of model.classes_), base of shape (3,), with your shapley_values, one output at a time."""
    raise NotImplementedError("shap_by_class_17")

In [ ]:
LR_17 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(XP_TRAIN, YP_TRAIN)   # references
RF_17 = RandomForestClassifier(n_estimators=200, random_state=0).fit(XP_TRAIN, YP_TRAIN)
G_17 = GENTOO_17[None]
with checked("B6.17"):
    lr_17 = fit_logistic_17(XP_TRAIN.copy(), YP_TRAIN.copy())
    if returned("B6.17", "fit_logistic_17", lr_17):
        steps_17 = [type(s).__name__ for _, s in getattr(lr_17, "steps", [])]
        verdict("B6.17", steps_17 == ["StandardScaler", "LogisticRegression"],
                "fit_logistic_17 : StandardScaler puis LogisticRegression.",
                f"fit_logistic_17 doit rendre le Pipeline (StandardScaler, LogisticRegression), reçu {steps_17 or type(lr_17).__name__}.")
        same("B6.17", "probabilités de fit_logistic_17 sur le test", lr_17.predict_proba(XP_TEST),
             LR_17.predict_proba(XP_TEST), rtol=1e-6, atol=1e-8)
    rf_17 = fit_forest_17(XP_TRAIN.copy(), YP_TRAIN.copy())
    if returned("B6.17", "fit_forest_17", rf_17):
        same("B6.17", "probabilités de fit_forest_17 sur le test", rf_17.predict_proba(XP_TEST), RF_17.predict_proba(XP_TEST))

with checked("B6.17"):
    f_p_17, f_l_17 = class_output_17(LR_17, 2, "proba"), class_output_17(LR_17, 2, "logit")
    if not (callable(f_p_17) and callable(f_l_17)):
        print("❌ Ex B6.17 : class_output_17 doit rendre une fonction f(X) -> (n,).")
        raise StopCheck
    same("B6.17", "class_output_17(LR, 2, 'proba') sur le test", f_p_17(XP_TEST), LR_17.predict_proba(XP_TEST)[:, 2])
    same("B6.17", "class_output_17(LR, 2, 'logit') sur le test", f_l_17(XP_TEST), LR_17.decision_function(XP_TEST)[:, 2])

RESULTS_17 = {}
with checked("B6.17"):
    phi_l_17, base_l_17 = pair("B6.17", "shap_by_class_17", shap_by_class_17(LR_17, GENTOO_17.copy(), XP_TRAIN.copy(), "logit"),
                               names=("phi", "base"))
    if np.shape(phi_l_17) != (3, 4) or np.shape(base_l_17) != (3,):
        print(f"❌ Ex B6.17 : attendu phi de forme (3, 4) et base de forme (3,), reçu {np.shape(phi_l_17)} et {np.shape(base_l_17)}.")
        raise StopCheck
    scaler_17, logreg_17 = LR_17[0], LR_17[-1]
    linear_17 = logreg_17.coef_ / scaler_17.scale_ * (GENTOO_17 - XP_TRAIN.mean(axis=0))   # w_ki (x_i - mean_i), ∂ B6.2
    if same("B6.17", "logits : phi = w_ki (x_i − moyenne du fond), le cas linéaire", phi_l_17, linear_17, rtol=1e-6, atol=1e-8):
        RESULTS_17["régression logistique, logit"] = np.asarray(phi_l_17)
    phi_p_17, base_p_17 = pair("B6.17", "shap_by_class_17", shap_by_class_17(LR_17, GENTOO_17.copy(), XP_TRAIN.copy(), "proba"),
                               names=("phi", "base"))
    ok_p_17 = same("B6.17", "probabilités : efficacité de chaque classe", np.sum(phi_p_17, axis=1) + base_p_17,
                   LR_17.predict_proba(G_17)[0], rtol=1e-6, atol=1e-8)
    ok_p_17 &= same("B6.17", "probabilités : valeurs de base = probabilités moyennes sur le fond", base_p_17,
                    LR_17.predict_proba(XP_TRAIN).mean(axis=0), rtol=1e-6, atol=1e-8)
    if ok_p_17:
        RESULTS_17["régression logistique, probabilité"] = np.asarray(phi_p_17)
    phi_rf_17, base_rf_17 = pair("B6.17", "shap_by_class_17", shap_by_class_17(RF_17, GENTOO_17.copy(), XP_TRAIN.copy(), "proba"),
                                 names=("phi", "base"))
    if shap is not None:                  # the whole background: TreeExplainer keeps at most 100 rows by default
        tree_17 = shap.TreeExplainer(RF_17, data=shap.maskers.Independent(XP_TRAIN, max_samples=len(XP_TRAIN)),
                                     feature_perturbation="interventional")(G_17)
        own_17 = float(np.abs(tree_17.values[0].sum(axis=0) + tree_17.base_values[0] - RF_17.predict_proba(G_17)[0]).max())
        print(f"ℹ️ forêt : TreeSHAP s'écarte lui-même de predict_proba de {own_17:.1e} (son efficacité) ; la comparaison "
              "se fait donc à 2e-3 près.")
        if same("B6.17", "forêt : tes valeurs contre TreeExplainer (interventionnel, fond complet)", phi_rf_17,
                tree_17.values[0].T, rtol=0, atol=2e-3):
            RESULTS_17["forêt aléatoire, probabilité"] = np.asarray(phi_rf_17)
if len(RESULTS_17) == 3:
    fig_17, axes_17 = plt.subplots(1, 3, figsize=(16, 3.8))
    for ax_17, (title_17, phi_17) in zip(axes_17, RESULTS_17.items()):
        for k_17, species_17 in enumerate(SPECIES):
            ax_17.bar(np.arange(4) + 0.27 * (k_17 - 1), phi_17[k_17], width=0.27, label=species_17)
        ax_17.axhline(0, color="black", lw=0.8)
        ax_17.set_xticks(range(4), [name.replace("_mm", "").replace("_g", "") for name in PENG.features], fontsize=8)
        ax_17.set_title(title_17, fontsize=10)
        ax_17.legend(fontsize=8)
    plt.suptitle("valeurs de Shapley du premier Gentoo du test, une sortie à la fois (fond : les 249 manchots d'entraînement)",
                 fontsize=10)
    plt.tight_layout()
    plt.show()
    for title_17, phi_17 in RESULTS_17.items():
        print(f"{title_17:>36} : somme sur les classes de chaque feature "
              + ", ".join(fr(0.0 if abs(v) < 5e-5 else v, 4) for v in phi_17.sum(axis=0)))

## Partie B · Réseaux : gradients intégrés et Grad-CAM

Tu écris d'abord en NumPy les deux méthodes de ta librairie qui ne dépendent pas du réseau : les gradients intégrés à partir d'une fonction de gradient, et la combinaison Grad-CAM à partir d'activations et de gradients déjà extraits (B6.18). Puis tu les branches sur des réseaux : saillance, gradient × entrée et gradients intégrés sur un petit CNN CIFAR-10 (B6.19), Grad-CAM par hooks sur le ResNet fine-tuné du B1, pour des bonnes prédictions et des erreurs (B6.20). Au ch. 21, FGSM et la maximisation d'activation dérivaient déjà par rapport à l'entrée (🔁 R1) ; au ch. 20 et au B1, tu as posé des hooks.

La cellule suivante charge 500 images de test de CIFAR-10 (`X_CIFAR`, `Y_CIFAR` : 50 par classe, en uint8) et `CNN`, le petit CNN de référence de B6.19 : trois blocs de deux convolutions, entraîné une fois par `wbx.cifar_cnn` (≈ 40 s sur CPU la première fois, puis relu du disque), **gelé** et en mode `eval()` ; il reçoit des pixels dans $[0, 1]$, de forme `(N, 3, 32, 32)`, et se normalise lui-même : l'image noire est le tenseur nul. `show_rows` affiche une grille d'images et de cartes.

In [ ]:
# Tools of part B: CIFAR-10 and the small CNN of B6.19 (the ResNet of B6.20 is loaded there)
import torch
import torch.nn.functional as F

CLASSES = wb.datasets.CIFAR10_CLASSES
X_CIFAR, Y_CIFAR = wb.datasets.load_cifar10_subset(50, "test")   # 500 test images, uint8 (N, 32, 32, 3), int labels
CNN, CNN_INFO = wbx.cifar_cnn(FAST_MODE)   # trained once (≈ 40 s on CPU), then read back; frozen, in eval() mode


def show_rows(rows, row_titles, col_titles, cmaps, size=1.7, suptitle=None):
    """A grid of images: rows[r][c] is an image (H, W, 3) in [0, 1] or a map (H, W); cmaps[c] the colour map of the
    column c ("gray" for the images is ignored), None for an image."""
    nrow, ncol = len(rows), len(rows[0])
    fig, axes = plt.subplots(nrow, ncol, figsize=(size * ncol, size * nrow + (0.5 if suptitle else 0.1)), squeeze=False)
    for r in range(nrow):
        for c in range(ncol):
            ax = axes[r, c]
            ax.imshow(rows[r][c], cmap=cmaps[c], vmin=None if cmaps[c] is None else 0)
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(col_titles[c], fontsize=8)
        axes[r, 0].set_ylabel(row_titles[r], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    plt.tight_layout()
    plt.show()


print(f"CIFAR-10 : {len(X_CIFAR)} images de test (50 par classe). Petit CNN de référence : "
      f"{sum(p.numel() for p in CNN.parameters())} paramètres, accuracy de test {pct(CNN_INFO['test_accuracy'], 1)}, "
      "entrée : des pixels dans [0, 1], (N, 3, 32, 32).")

### Ex B6.18 — integrated_gradients et grad_cam 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire les gradients intégrés à partir d'une fonction de gradient, et la combinaison Grad-CAM de cartes et de gradients déjà extraits.  
**Prérequis :** ✏️ B6.3, B6.4 · fiche §206.5 (« Les gradients intégrés », « En pratique : une somme à la place de l'intégrale ») et §206.6 (« Un exemple à la main », « Avec PyTorch : deux hooks ») · **mylearn :** `explain.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/explain.py` (créé par `python tools/start_chapter.py B6`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb6_explain.py -q` les lance tous dans un terminal.

**Contrat** de `integrated_gradients(grad_fn, x, baseline=None, n_steps=50, method="trapezoid")` :
- les points du chemin sont $\mathbf{x}' + \alpha(\mathbf{x} - \mathbf{x}')$, $\mathbf{x}'$ la ligne de référence `baseline` (des zéros si `None`) ; la moyenne pondérée des gradients est multipliée par $\mathbf{x} - \mathbf{x}'$, et le résultat a la forme de `x`, en float, même pour un `x` entier ;
- `n_steps` compte les **intervalles**, un entier ≥ 1 : `'left'`, `'right'` et `'midpoint'` utilisent $n$ points de poids $1/n$, `'trapezoid'` les $n + 1$ points $k/n$ de poids $1/n$, divisés par deux aux extrémités (la docstring) ;
- `grad_fn` ne reçoit que des **batches** de forme `(n, *x.shape)`, même avec `n_steps = 1` (pour un seul point : `point[None]`, puis `[0]` sur le résultat) ; elle peut être appelée une fois ou sur plusieurs morceaux ;
- `ValueError` si `n_steps < 1`, si `method` est inconnue (les noms de captum, comme `'riemann_trapezoid'`, sont refusés), ou si `baseline` n'a pas **exactement** la forme de `x` (une forme seulement diffusable est refusée) ; `x` et `baseline` ne sont pas modifiés ;
- ⚠️ la note de la docstring sur captum est inexacte : son `riemann_trapezoid` à $n + 1$ points vaut ton `'trapezoid'` **multiplié par $n/(n+1)$**, parce que ses poids somment à moins de 1 (fiche, 🕰️ de 206.6 ; ✏️ B6.3 k) ; `'left'`, `'right'` et `'midpoint'` coïncident avec `riemann_left`, `riemann_right` et `riemann_middle` au même nombre de points.

**Contrat** de `grad_cam(activations, gradients, relu=True, normalize=True)` :
- les deux entrées ont exactement la même forme 4-D `(N, C, H, W)`, sinon `ValueError` ; le résultat a la forme `(N, H, W)` ;
- le poids d'un canal est la **moyenne** spatiale de ses gradients ; la carte est la somme pondérée des canaux, la ReLU appliquée **une fois**, à cette somme (`relu=True`) ;
- `normalize=True` divise **chaque** carte par sa propre plus grande valeur absolue, sans epsilon ; une carte nulle, y compris après la ReLU, reste nulle (pas de NaN) ;
- les entrées ne sont pas modifiées.

La première cellule de vérification essaie `integrated_gradients` sur les exemples de la docstring, sur $x_0 x_1 + x_2^3$ en $(1, 2, -1)$ avec les quatre règles et 6 intervalles (formes fermées), sur ce que reçoit `grad_fn`, sur la complétude de la sigmoïde de 206.5, un `x` entier et trois refus ; la seconde partie essaie `grad_cam` sur la docstring, sur trois images de 4 canaux (contre `np.einsum`), l'invariance à un facteur positif des gradients, `relu=False`, des gradients nuls et deux refus. La seconde cellule lance les 111 tests de l'exercice (`-k "test_integrated_gradients_ or test_grad_cam_"`) ; sans captum, 12 sont ignorés.

Dans tes notes : avec 6 intervalles, quelle règle donne la valeur exacte pour $x_0 x_1$, et pourquoi pas pour $x_2^3$ ? Pourquoi la complétude sert-elle de test à une implémentation ? Pourquoi `grad_cam` ne fait-elle que la combinaison, et pas l'extraction (qui reste côté PyTorch) ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B6")   # reload your saved file
with checked("B6.18"):
    E = mylearn.explain
    grad_q_18 = lambda X: np.stack([2 * X[:, 0], np.full(len(X), 3.0)], axis=1)       # f = x0**2 + 3 x1 (docstring)
    same("B6.18", "docstring : x0² + 3 x1 en (1, 1), n_steps=4, 'left'",
         E.integrated_gradients(grad_q_18, [1.0, 1.0], n_steps=4, method="left"), [0.75, 3.0])
    same("B6.18", "docstring : la même avec 'trapezoid'", E.integrated_gradients(grad_q_18, [1.0, 1.0], n_steps=4), [1.0, 3.0])

    # f = x0 x1 + x2**3 at x = (1, 2, -1) from a zero baseline: on the path a x, the gradient is (a x1, a x0, 3 a² x2²)
    grad_c_18 = lambda X: np.stack([X[:, 1], X[:, 0], 3 * X[:, 2] ** 2], axis=1)
    x_18 = np.array([1.0, 2.0, -1.0])
    k_18 = np.arange(7) / 6                                       # n = 6 intervals
    means_18 = {"left": (k_18[:-1].mean(), (k_18[:-1] ** 2).mean()), "right": (k_18[1:].mean(), (k_18[1:] ** 2).mean()),
                "midpoint": (((np.arange(6) + 0.5) / 6).mean(), (((np.arange(6) + 0.5) / 6) ** 2).mean()),
                "trapezoid": (0.5, 1 / 3 + 1 / (6 * 36))}         # the means of a and a² for each rule
    for method_18, (m1_18, m2_18) in means_18.items():
        want_18 = np.array([x_18[0] * x_18[1] * m1_18, x_18[1] * x_18[0] * m1_18, x_18[2] * 3 * x_18[2] ** 2 * m2_18])
        same("B6.18", f"x0 x1 + x2³ en (1, 2, −1), n_steps=6, '{method_18}'",
             E.integrated_gradients(grad_c_18, x_18.copy(), n_steps=6, method=method_18), want_18)
    shapes_18 = []

    def spy_18(X):
        shapes_18.append(np.shape(X))
        return grad_c_18(np.asarray(X, dtype=float))

    E.integrated_gradients(spy_18, x_18, n_steps=1, method="left")
    verdict("B6.18", shapes_18 and all(len(s) == 2 and s[1] == 3 for s in shapes_18),
            "grad_fn ne reçoit que des batches (n, 3), même avec n_steps=1.",
            f"grad_fn a reçu les formes {shapes_18} : elle attend un batch (n, *x.shape), même pour un seul point.")
    sig_18 = lambda X: (lambda s: np.stack([s * (1 - s) * X[:, 1], s * (1 - s) * X[:, 0]], axis=1))(
        1 / (1 + np.exp(-(X[:, 0] * X[:, 1] - 2))))                # the sigmoid of 206.5: sigma(x0 x1 - 2)
    gap_18 = 1 / (1 + np.exp(-4.0)) - 1 / (1 + np.exp(2.0))
    total_18 = float(np.sum(E.integrated_gradients(sig_18, [2.0, 3.0], n_steps=200)))
    verdict("B6.18", abs(total_18 - gap_18) < 1e-4, f"complétude sur la sigmoïde de 206.5 (200 intervalles) : somme {fr(total_18, 5)}.",
            f"complétude sur la sigmoïde de 206.5 : la somme vaut {total_18:.5g}, attendu f(x) − f(x′) ≈ {gap_18:.5g}.")
    ig_int_18 = E.integrated_gradients(grad_q_18, np.array([1, 1]), n_steps=4)
    verdict("B6.18", np.asarray(ig_int_18).dtype.kind == "f", "un x entier donne des attributions en float.",
            f"un x entier doit donner des attributions en float, reçu {np.asarray(ig_int_18).dtype}.")
    refuses("B6.18", "n_steps=0", lambda: E.integrated_gradients(grad_q_18, [1.0, 1.0], n_steps=0))
    refuses("B6.18", "method='riemann_trapezoid' (un nom de captum)",
            lambda: E.integrated_gradients(grad_q_18, [1.0, 1.0], method="riemann_trapezoid"))
    refuses("B6.18", "une ligne de référence de forme (1,) pour x de forme (2,)",
            lambda: E.integrated_gradients(grad_q_18, [1.0, 1.0], baseline=[0.0]))

with checked("B6.18"):
    E = mylearn.explain
    A_18 = np.array([[[[1.0, 2.0], [0.0, 1.0]], [[0.0, 1.0], [3.0, 0.0]]]])           # the docstring: N=1, C=2, 2 x 2
    G_18 = np.array([[[[1.0, 1.0], [1.0, 1.0]], [[-1.0, -1.0], [-1.0, -1.0]]]])
    copies_18 = (A_18.copy(), G_18.copy())
    same("B6.18", "docstring : grad_cam(A, G, relu=False, normalize=False)", E.grad_cam(A_18, G_18, relu=False, normalize=False),
         [[[1.0, 1.0], [-3.0, 1.0]]])
    same("B6.18", "docstring : grad_cam(A, G)", E.grad_cam(A_18, G_18), [[[1.0, 1.0], [0.0, 1.0]]])
    verdict("B6.18", np.array_equal(A_18, copies_18[0]) and np.array_equal(G_18, copies_18[1]),
            "les activations et les gradients reçus n'ont pas changé.", "grad_cam a modifié ses entrées.")
    rng_18 = np.random.default_rng(18)
    A2_18, G2_18 = rng_18.random((3, 4, 5, 5)), rng_18.normal(size=(3, 4, 5, 5))
    want_raw_18 = np.einsum("nc,nchw->nhw", G2_18.mean(axis=(2, 3)), A2_18)
    want_18 = np.maximum(want_raw_18, 0)
    peak_18 = want_18.max(axis=(1, 2), keepdims=True)
    same("B6.18", "3 images de 4 canaux 5 × 5 : ReLU de la somme pondérée, chaque carte ÷ son maximum",
         E.grad_cam(A2_18, G2_18), np.divide(want_18, peak_18, out=np.zeros_like(want_18), where=peak_18 > 0))
    same("B6.18", "les mêmes, gradients multipliés par 7 (même carte)", E.grad_cam(A2_18, 7 * G2_18),
         np.divide(want_18, peak_18, out=np.zeros_like(want_18), where=peak_18 > 0))
    same("B6.18", "relu=False : dans [−1, 1], ÷ le plus grand |.|", E.grad_cam(A2_18, G2_18, relu=False),
         want_raw_18 / np.abs(want_raw_18).max(axis=(1, 2), keepdims=True))
    same("B6.18", "gradients nuls : une carte nulle, sans NaN", E.grad_cam(A2_18, np.zeros_like(G2_18)), np.zeros((3, 5, 5)))
    refuses("B6.18", "activations et gradients de formes différentes", lambda: E.grad_cam(A2_18, G2_18[:, :3]))
    refuses("B6.18", "des tableaux 3-D", lambda: E.grad_cam(A2_18[0], G2_18[0]))

In [ ]:
if written(lambda: (mylearn.explain.integrated_gradients(lambda X: X, [1.0], n_steps=1), mylearn.explain.grad_cam(np.ones((1, 1, 1, 1)), np.ones((1, 1, 1, 1))))):
    run_pytest_group(TEST_EXPLAIN, "test_integrated_gradients_ or test_grad_cam_")   # 111 tests, dont 12 ignorés sans captum
else:
    print("⏳ Ex B6.18 : les tests attendent tes fonctions.")

### Ex B6.19 — Saillance, gradient × entrée et gradients intégrés sur un CNN CIFAR-10 📦 ★★ ⏱️ 30 min
**Objectif :** calculer trois attributions pixel à pixel sur un CNN, vérifier la complétude et comparer deux lignes de référence.  
**Prérequis :** Ex B6.18 · ch. 24 · fiche §206.5 (« Le gradient comme explication », « Les gradients intégrés », « Le choix de la ligne de référence ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

La cellule donnée prend 8 images de test (`IMAGES_19`, un tenseur `(8, 3, 32, 32)` dans $[0, 1]$), leurs classes (`LABELS_19`), les classes que prédit le CNN (`PRED_19`) et `blurred_19`, qui floute une image (la moyenne sur une fenêtre de 7 × 7 pixels). **Contrat** :
- `class_gradient_19(model, images, target)` rend un tableau NumPy `(N, 3, 32, 32)` : le gradient du **logit** (le score avant la softmax) de la classe `target[n]` de chaque image par rapport à ses pixels ; `images` est un tenseur ou un tableau `(N, 3, 32, 32)`, `target` une classe par image, pas forcément la prédite ; le modèle est gelé et en mode `eval()`, et doit le rester ; deux appels successifs rendent le même gradient ;
- `saliency_19(gradients)` rend la carte de saillance `(N, 32, 32)` : le plus grand $\lvert$gradient$\rvert$ des trois canaux de chaque pixel ;
- `grad_times_input_19(gradients, images)` rend `(N, 32, 32)` : gradient × entrée, sommé sur les trois canaux de chaque pixel, avec son signe ;
- `integrated_19(model, image, target, baseline, n_steps=50)` rend `(3, 32, 32)` : les gradients intégrés du logit de la classe `target` (un entier) pour **une** image `(3, 32, 32)`, depuis la ligne de référence `baseline` `(3, 32, 32)`, calculés par **ta** `integrated_gradients` (règle `'trapezoid'`) avec une `grad_fn` construite sur `class_gradient_19`.

La cellule de vérification compare chaque fonction à une référence (l'autograd de `wbx`) sur quatre images, avec des classes que le CNN ne prédit pas. L'expérience trace, pour 6 images et la vraie classe, la saillance, $\lvert$gradient × entrée$\rvert$, et $\lvert$IG$\rvert$ depuis l'image noire et depuis l'image floutée (la somme des valeurs absolues des trois canaux), puis compare la somme des attributions à $f(\mathbf{x}) - f(\mathbf{x}')$.

Dans tes notes : laquelle des trois cartes est la plus bruitée, et laquelle se concentre le plus sur l'objet ? La complétude tient-elle pour les gradients intégrés, et pour gradient × entrée ? Avec 50 intervalles, à combien près ? Que change la ligne de référence floutée : que vaut $f(\mathbf{x}')$, quels pixels reçoivent une attribution presque nulle, et pourquoi (le facteur $x_i - x'_i$) ? Quelle référence choisirais-tu pour un objet noir sur fond clair ?

In [ ]:
IMAGES_19 = wbx.pixels(X_CIFAR[:8])               # 8 test images, float32 tensor (8, 3, 32, 32) in [0, 1]
LABELS_19 = Y_CIFAR[:8]
PRED_19 = wbx.predict(CNN, IMAGES_19).argmax(axis=1)  # the classes the CNN predicts (computed without your code)


def blurred_19(x):
    """A blurred copy of images (N, 3, 32, 32): the mean over a 7 x 7 window around each pixel."""
    return F.avg_pool2d(torch.as_tensor(x), 7, stride=1, padding=3, count_include_pad=False)


print("classes vraies :", [CLASSES[k] for k in LABELS_19])
print("classes prédites :", [CLASSES[k] for k in PRED_19])

In [ ]:
def class_gradient_19(model, images, target):
    """np.ndarray (N, 3, 32, 32): the gradient of the logit of the class target[n] of each image with respect to its
    pixels. images: a float tensor or array (N, 3, 32, 32) in [0, 1]; target: one class per image (N ints)."""
    raise NotImplementedError("class_gradient_19")


def saliency_19(gradients):
    """(N, 32, 32): the saliency map, the largest |gradient| over the 3 channels of each pixel."""
    raise NotImplementedError("saliency_19")


def grad_times_input_19(gradients, images):
    """(N, 32, 32): gradient × input, summed over the 3 channels of each pixel (signed)."""
    raise NotImplementedError("grad_times_input_19")


def integrated_19(model, image, target, baseline, n_steps=50):
    """(3, 32, 32): the integrated gradients of the logit of the class `target` (an int) for ONE image (3, 32, 32),
    from `baseline` (3, 32, 32), with your integrated_gradients (rule 'trapezoid') and a grad_fn built on
    class_gradient_19."""
    raise NotImplementedError("integrated_19")

In [ ]:
ok_19 = {}
with checked("B6.19"):
    x_19 = IMAGES_19[:4]
    t_19 = np.array([(p + 3) % 10 for p in PRED_19[:4]])     # classes that the CNN does NOT predict
    want_19 = wbx.input_gradients(CNN, x_19, t_19)
    g_19 = class_gradient_19(CNN, x_19.clone(), t_19.copy())
    if CNN.training or any(p.requires_grad for p in CNN.parameters()):
        print("❌ Ex B6.19 : après class_gradient_19, le modèle n'est plus gelé ou n'est plus en mode eval() (ses "
              "batchnorms changent alors le calcul) : ta fonction ne doit pas le changer.")
        raise StopCheck
    if not returned("B6.19", "class_gradient_19", g_19):
        raise StopCheck
    arr_19 = np.asarray(g_19.detach() if torch.is_tensor(g_19) else g_19, dtype=float)
    if arr_19.shape != want_19.shape:
        print(f"❌ Ex B6.19 : class_gradient_19 doit rendre un tableau (4, 3, 32, 32), reçu la forme {arr_19.shape}.")
        raise StopCheck
    if not np.allclose(arr_19, want_19, rtol=1e-4, atol=1e-6):
        probe_19 = x_19.clone().requires_grad_(True)
        probs_19 = torch.softmax(CNN(probe_19), dim=1)
        (g_prob_19,) = torch.autograd.grad(probs_19.gather(1, torch.as_tensor(t_19)[:, None]).sum(), probe_19)
        if np.allclose(arr_19, wbx.input_gradients(CNN, x_19, PRED_19[:4]), rtol=1e-4, atol=1e-6):
            print("❌ Ex B6.19 : c'est le gradient de la classe prédite : le contrat veut celui de target[n], qui n'est "
                  "pas forcément la prédite.")
        elif np.allclose(arr_19, g_prob_19.numpy(), rtol=1e-4, atol=1e-6):
            print("❌ Ex B6.19 : c'est le gradient de la probabilité (après la softmax) : le contrat veut celui du logit.")
        elif np.allclose(arr_19, sum(wbx.input_gradients(CNN, x_19, k) for k in range(10)), rtol=1e-4, atol=1e-5):
            print("❌ Ex B6.19 : c'est le gradient de la somme des 10 logits : ne garde que celui de target[n].")
        else:
            same("B6.19", "class_gradient_19 contre l'autograd de référence", arr_19, want_19, rtol=1e-4, atol=1e-6)
        raise StopCheck
    print("✅ Ex B6.19 : class_gradient_19 : le gradient du logit de la classe demandée, image par image.")
    again_19 = np.asarray(class_gradient_19(CNN, x_19.clone(), t_19.copy()), dtype=float)
    verdict("B6.19", np.allclose(again_19, arr_19, rtol=1e-6, atol=1e-8), "un second appel rend le même gradient (rien ne s'accumule).",
            "un second appel ne rend pas le même gradient : un .grad s'accumule-t-il d'un appel à l'autre ?")
    print("✅ Ex B6.19 : le modèle reste gelé, en mode eval().")
    ok_19["grad"] = True
CNN.eval().requires_grad_(False)                       # restored for the cells that follow, whatever your code did

with checked("B6.19"):
    if not ok_19:
        raise StopCheck
    s_19 = saliency_19(want_19.copy())
    if same("B6.19", "saliency_19 : le plus grand |gradient| des 3 canaux", s_19, np.abs(want_19).max(axis=1)):
        ok_19["saliency"] = True
    elif np.allclose(np.asarray(s_19, dtype=float), np.abs(want_19).mean(axis=1)):
        print("❌ Ex B6.19 : c'est la moyenne des |gradients| des canaux : la carte de saillance de 206.5 prend le maximum.")
    gi_19 = grad_times_input_19(want_19.copy(), x_19.numpy().copy())
    if same("B6.19", "grad_times_input_19 : somme sur les canaux de gradient × pixel", gi_19, (want_19 * x_19.numpy()).sum(axis=1),
            rtol=1e-6, atol=1e-8):
        ok_19["gxi"] = True

with checked("B6.19"):
    if not ok_19.get("grad"):
        raise StopCheck
    ig_19 = integrated_19(CNN, IMAGES_19[0].numpy(), int(LABELS_19[0]), np.zeros((3, 32, 32)), n_steps=20)
    if not returned("B6.19", "integrated_19", ig_19):
        raise StopCheck
    ref_ig_19 = wbx.integrated_gradients(CNN, IMAGES_19[:1], [int(LABELS_19[0])], n_steps=20)[0]
    if same("B6.19", "integrated_19 (image noire, 20 intervalles) contre la référence", ig_19, ref_ig_19, rtol=1e-4, atol=1e-6):
        ok_19["ig"] = True

In [ ]:
if len(ok_19) == 4:
    start_19 = time.perf_counter()
    black_19, blur_19 = torch.zeros_like(IMAGES_19), blurred_19(IMAGES_19)
    grads_19 = class_gradient_19(CNN, IMAGES_19, LABELS_19)
    sal_19, gxi_19 = saliency_19(grads_19), grad_times_input_19(grads_19, IMAGES_19.numpy())
    ig_black_19 = np.stack([integrated_19(CNN, IMAGES_19[i].numpy(), int(LABELS_19[i]), black_19[i].numpy())
                            for i in range(6)])
    ig_blur_19 = np.stack([integrated_19(CNN, IMAGES_19[i].numpy(), int(LABELS_19[i]), blur_19[i].numpy())
                           for i in range(6)])
    rows_19 = []
    for i in range(6):
        rows_19.append([IMAGES_19[i].permute(1, 2, 0).numpy(), sal_19[i], np.abs(gxi_19[i]),
                        np.abs(ig_black_19[i]).sum(axis=0), blur_19[i].permute(1, 2, 0).numpy(),
                        np.abs(ig_blur_19[i]).sum(axis=0)])
    show_rows(rows_19, [f"{CLASSES[LABELS_19[i]]} (prédit {CLASSES[PRED_19[i]]})" for i in range(6)],
              ["image", "saillance", "|gradient × entrée|", "|IG|, réf. noire", "réf. floutée", "|IG|, réf. floutée"],
              [None, "magma", "magma", "magma", None, "magma"],
              suptitle="attributions du logit de la vraie classe (IG : somme des |attributions| sur les 3 canaux)")
    with torch.no_grad():
        logit_19 = lambda x: CNN(x).gather(1, torch.as_tensor(LABELS_19[:6])[:, None])[:, 0].double().numpy()
        f_x_19, f_black_19, f_blur_19 = logit_19(IMAGES_19[:6]), logit_19(black_19[:6]), logit_19(blur_19[:6])
    print("complétude (somme des attributions contre f(x) − f(x′), logit de la vraie classe) :")
    for i in range(6):
        print(f"  {CLASSES[LABELS_19[i]]:>10} : réf. noire {fr(ig_black_19[i].sum(), 3)} contre {fr(f_x_19[i] - f_black_19[i], 3)} ; "
              f"réf. floutée {fr(ig_blur_19[i].sum(), 3)} contre {fr(f_x_19[i] - f_blur_19[i], 3)} ; "
              f"somme de gradient × entrée {fr(gxi_19[i].sum(), 3)}")
    print(f"ℹ️ {time.perf_counter() - start_19:.1f} s")
else:
    print("⏳ Ex B6.19 : l'expérience attend tes quatre fonctions (cellule précédente).")

### Ex B6.20 — Grad-CAM par hooks sur le ResNet fine-tuné du B1 : bonnes prédictions et erreurs 📦 ★★★ ⏱️ 45 min
**Objectif :** extraire activations et gradients par deux hooks, construire les cartes Grad-CAM d'un ResNet et les lire sur des bonnes prédictions et des erreurs.  
**Prérequis :** Ex B6.18 · B1 (B1.17, B1.27) · fiche §206.6 (« Avec PyTorch : deux hooks ») · ch. 20 (hooks) · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

Le modèle est **ton** ResNet-18 du B1 si tu l'as publié en B1.27 : `wbx.find_b1_folder()` cherche le dossier `resnet18_cifar10_b1` (avec `model.safetensors` et `config.json`) à côté de tes notebooks de `mon_travail/` ; sinon, ou si tu mets `B1_FOLDER = None`, c'est un ResNet-18 de référence, fine-tuné une fois par la recette de B1.17 (`layer4` et une nouvelle tête, 100 images par classe à 96 pixels, 3 epochs : ≈ 35 s sur CPU la première fois, puis relu du disque). La cellule de chargement dit lequel est utilisé. Le modèle est **gelé** (`requires_grad=False` partout) et en mode `eval()`, comme un backbone du B1 ; il attend des images préparées par `wbx.model_input` (agrandies à `SIZE_20` pixels, normalisées par les statistiques d'ImageNet), et `layer4` y est en `SIZE_20 // 32` × `SIZE_20 // 32` (3 × 3 à 96 pixels). La cellule de chargement choisit aussi, parmi 200 images de test, 8 bonnes prédictions et 8 erreurs (`X_20`, 16 images préparées).

**Contrat** :
- `activations_and_gradients_20(model, layer, images, target)` rend le couple `(A, G)` de tableaux NumPy `(N, C, H, W)` : `A`, la sortie du module `layer` pour le batch préparé `images`, et `G`, le gradient de la somme, sur les images, du logit de `target[n]` par rapport à `A` (une classe par image, pas forcément la prédite) ; deux hooks, l'un sur le module (passe avant), l'autre sur son tenseur de sortie ; le hook de passe avant ne rend rien, et il est retiré avant de rendre la main, même en cas d'erreur ;
- `cam_20(model, layer, images, target, size)` rend `(N, size, size)` : les cartes de **ta** `grad_cam` (ReLU et normalisation par défaut), agrandies par `F.interpolate(..., mode="bilinear", align_corners=False)`, la convention qui a agrandi les images.

La cellule de vérification compare `A`, `G` et les cartes à une référence (`wbx`) sur 4 images, dont une avec une autre classe que la vraie, et vérifie qu'aucun hook ne reste posé. L'expérience trace la carte de la classe prédite sur les 8 bonnes prédictions, puis, sur les 8 erreurs, celle de la classe prédite et celle de la vraie classe, et affiche la corrélation entre ces deux cartes.

Dans tes notes : sur les bonnes prédictions, la carte tombe-t-elle sur l'objet, sur le fond, sur un bord ? Sur les erreurs, les cartes de la classe prédite et de la vraie classe regardent-elles au même endroit, et qu'est-ce que cela suggère de l'erreur (un objet ambigu, un contexte trompeur) ? Que peut-on localiser avec une carte de 3 × 3 sur une image de 96 pixels ? Que verrais-tu avec `layer3`, et pourquoi Grad-CAM vise-t-il la dernière couche convolutive (🧠 Q8) ? Une carte qui « tombe sur l'objet » prouve-t-elle que le modèle a raison pour de bonnes raisons (206.7) ?

In [ ]:
B1_FOLDER = wbx.find_b1_folder()   # your model published in Ex B1.27, if it is there (None: the reference ResNet)


def activations_and_gradients_20(model, layer, images, target):
    """(A, G): np.ndarray (N, C, H, W) twice, the output A of `layer` (a module of model) for the prepared batch
    `images`, and the gradient G of the sum over the images of the logit of target[n] with respect to A; two hooks
    (one on the module, one on its output tensor), removed before returning."""
    raise NotImplementedError("activations_and_gradients_20")


def cam_20(model, layer, images, target, size):
    """(N, size, size): the Grad-CAM maps of your grad_cam (relu and normalize by default), resized to size x size by
    F.interpolate(..., mode="bilinear", align_corners=False)."""
    raise NotImplementedError("cam_20")

In [ ]:
RESNET, RESNET_INFO = wbx.b1_resnet(FAST_MODE, folder=B1_FOLDER)   # frozen, in eval() mode
SIZE_20 = RESNET_INFO["size"]                                          # the side of its input images (96 in fast mode)
POOL_20 = X_CIFAR[:200], Y_CIFAR[:200]                                 # 200 test images (20 per class)
PRED_20 = wbx.predict(RESNET, POOL_20[0], RESNET_INFO).argmax(axis=1)
RIGHT_20 = np.flatnonzero(PRED_20 == POOL_20[1])[:8]                   # 8 well-classified images
WRONG_20 = np.flatnonzero(PRED_20 != POOL_20[1])[:8]                   # 8 errors
X_20 = wbx.model_input(POOL_20[0][np.concatenate([RIGHT_20, WRONG_20])], RESNET_INFO)   # (16, 3, SIZE_20, SIZE_20)
print(f"ResNet-18 {'de référence' if RESNET_INFO['source'] == 'reference' else 'du B1 (le tien)'} : {SIZE_20} pixels, layer4 en {SIZE_20 // 32} × {SIZE_20 // 32} ; accuracy sur "
      f"les 200 images : {pct(np.mean(PRED_20 == POOL_20[1]), 1)} ; 8 bonnes prédictions et 8 erreurs retenues.")

In [ ]:
ok_20 = {}
RESNET.layer4._forward_hooks.clear()                   # hooks left by an earlier run of your code
with checked("B6.20"):
    x_20 = X_20[[0, 1, 8, 9]]
    true_20 = POOL_20[1][np.concatenate([RIGHT_20, WRONG_20])][[0, 1, 8, 9]]
    t_20 = np.array([true_20[0], (true_20[1] + 1) % 10, true_20[2], true_20[3]])      # true classes, and one other
    A_ref_20, G_ref_20, _ = wbx.activations_and_gradients(RESNET, RESNET.layer4, x_20, t_20)
    try:
        got_20 = activations_and_gradients_20(RESNET, RESNET.layer4, x_20.clone(), t_20.copy())
    except RuntimeError as exc_20:
        if "require" in str(exc_20) and "grad" in str(exc_20):
            print("❌ Ex B6.20 : ton code lève RuntimeError : " + str(exc_20)[:160] + " ; les poids du modèle sont gelés "
                  "(requires_grad=False) : relis le dernier détail de « Avec PyTorch : deux hooks » (§206.6).")
            raise StopCheck
        raise
    A_20, G_20 = pair("B6.20", "activations_and_gradients_20", got_20, names=("A", "G"))
    if RESNET.layer4._forward_hooks:
        print("❌ Ex B6.20 : un hook reste posé sur layer4 après l'appel : retire-le (handle.remove(), dans un finally) ; "
              "sinon chaque passe avant suivante le déclenche encore.")
        RESNET.layer4._forward_hooks.clear()
        raise StopCheck
    if same("B6.20", "A : la sortie de layer4", A_20, A_ref_20, rtol=1e-5, atol=1e-6):
        ok_20["A"] = True
    if np.shape(G_20) == G_ref_20.shape and not np.allclose(np.asarray(G_20, dtype=float), G_ref_20, rtol=1e-4, atol=1e-7):
        pred_t_20 = wbx.predict(RESNET, x_20).argmax(axis=1)
        _, G_pred_20, _ = wbx.activations_and_gradients(RESNET, RESNET.layer4, x_20, pred_t_20)
        if np.allclose(np.asarray(G_20, dtype=float), G_pred_20, rtol=1e-4, atol=1e-7):
            print("❌ Ex B6.20 : G est le gradient de la classe prédite : le contrat veut celui de target[n] "
                  "(l'image 2 reçoit une autre classe).")
            raise StopCheck
    if same("B6.20", "G : le gradient du logit de target[n] par rapport à A", G_20, G_ref_20, rtol=1e-4, atol=1e-7):
        ok_20["G"] = True
RESNET.eval().requires_grad_(False)                    # restored for the cells that follow, whatever your code did
RESNET.layer4._forward_hooks.clear()

with checked("B6.20"):
    if len(ok_20) < 2:
        raise StopCheck
    cams_20 = cam_20(RESNET, RESNET.layer4, x_20.clone(), t_20.copy(), SIZE_20)
    want_cam_20 = wbx.grad_cam(RESNET, RESNET.layer4, x_20, t_20, size=SIZE_20)
    arr_cam_20 = np.asarray(cams_20, dtype=float)
    if arr_cam_20.shape == want_cam_20.shape and not np.allclose(arr_cam_20, want_cam_20, rtol=1e-4, atol=1e-5):
        small_20 = torch.as_tensor(wbx.grad_cam(RESNET, RESNET.layer4, x_20, t_20))[:, None].float()
        corners_20 = F.interpolate(small_20, size=(SIZE_20, SIZE_20), mode="bilinear", align_corners=True)[:, 0].numpy()
        if np.allclose(arr_cam_20, corners_20, rtol=1e-4, atol=1e-5):
            print("❌ Ex B6.20 : la carte est agrandie avec align_corners=True : les images le sont avec "
                  "align_corners=False (wb.transfer.to_input) ; avec deux conventions, la carte est décalée sur l'image.")
            raise StopCheck
        if arr_cam_20.min() < -1e-9:
            print("❌ Ex B6.20 : la carte a des valeurs négatives : grad_cam garde-t-il sa ReLU (relu=True par défaut) ?")
            raise StopCheck
    if same("B6.20", f"cam_20 : cartes agrandies à {SIZE_20} × {SIZE_20}", arr_cam_20, want_cam_20, rtol=1e-4, atol=1e-5):
        ok_20["cam"] = True
RESNET.layer4._forward_hooks.clear()

In [ ]:
if len(ok_20) == 3:
    labels_20 = POOL_20[1][np.concatenate([RIGHT_20, WRONG_20])]
    preds_20 = PRED_20[np.concatenate([RIGHT_20, WRONG_20])]
    cam_pred_20 = cam_20(RESNET, RESNET.layer4, X_20, preds_20, SIZE_20)        # the predicted class
    cam_true_20 = cam_20(RESNET, RESNET.layer4, X_20, labels_20, SIZE_20)       # the true class (the same when right)
    shown_20 = wbx.model_input(POOL_20[0][np.concatenate([RIGHT_20, WRONG_20])], {"size": SIZE_20, "mean": (0, 0, 0),
                                                                                  "std": (1, 1, 1)})
    pics_20 = shown_20.permute(0, 2, 3, 1).clamp(0, 1).numpy()                  # the images at SIZE_20 pixels, in [0, 1]

    def overlay_20(image, cam):
        heat = plt.cm.jet(cam)[..., :3]
        return np.clip(0.55 * image + 0.45 * heat, 0, 1)

    def corr_20(a, b):
        """The correlation of two maps, or None when one of them is constant (all zero after the ReLU)."""
        a, b = a.ravel(), b.ravel()
        return None if a.std() == 0 or b.std() == 0 else float(np.corrcoef(a, b)[0, 1])

    rows_20 = [[pics_20[i], overlay_20(pics_20[i], cam_pred_20[i])] for i in range(8)]
    show_rows(rows_20, [f"{CLASSES[labels_20[i]]}" for i in range(8)], ["image", "Grad-CAM (prédite)"],
              [None, None], size=1.6, suptitle="bonnes prédictions : Grad-CAM de la classe prédite (layer4)")
    rows_w_20 = [[pics_20[i], overlay_20(pics_20[i], cam_pred_20[i]), overlay_20(pics_20[i], cam_true_20[i])]
                 for i in range(8, 16)]
    show_rows(rows_w_20, [f"{CLASSES[labels_20[i]]} → {CLASSES[preds_20[i]]}" for i in range(8, 16)],
              ["image", "classe prédite", "vraie classe"], [None, None, None], size=1.6,
              suptitle="erreurs (vraie → prédite) : Grad-CAM de la classe prédite, puis de la vraie classe")
    same_20 = [corr_20(cam_pred_20[i], cam_true_20[i]) for i in range(8, 16)]
    print("erreurs : corrélation entre la carte de la classe prédite et celle de la vraie classe :",
          ", ".join("carte nulle" if v is None else fr(v, 2) for v in same_20))
    empty_20 = [f"{CLASSES[labels_20[i]]} ({which})" for i in range(16)
                for which, cams in (("classe prédite", cam_pred_20), ("vraie classe", cam_true_20)) if cams[i].max() == 0]
    if empty_20:
        print("cartes nulles (la ReLU a tout éteint) :", ", ".join(empty_20))
else:
    print("⏳ Ex B6.20 : l'expérience attend tes deux fonctions (cellules précédentes).")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main les valeurs de Shapley d'un petit jeu, avec un fond, et dire ce que le fond change ?
2. Sais-tu calculer des gradients intégrés et une carte Grad-CAM à la main, et vérifier la complétude ?
3. Sais-tu auditer un classifieur par groupe (taux de sélection, TPR, FPR, precision, écarts) et expliquer pourquoi tous les critères ne peuvent pas tenir ensemble ?
4. Sais-tu dire ce qu'un scoring de crédit implique au regard du RGPD et de l'AI Act ?

**Pour aller plus loin** : l'article de Lundberg et Lee et le livre en ligne de C. Molnar, cités dans la fiche. La suite : le bonus B7, le MLOps.

*Les exercices B6.21 à B6.29 (un Grad-CAM à réparer, le test de santé, l'équité) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B6` pour obtenir le notebook complet.*